# What Is Knowable About Month-Ahead European Gas Prices — Full Results

This notebook reproduces every result in the umbrella paper, running the component studies in paper order: **§3.3** the Gregory–Hansen endogenous-break test, **§4 European** (core / Conditional I & II / lean), **§5 Global** (Compact / conditional / lean / ECM / VECM), **§6 Volatility** (GARCH), and **§7 Machine learning** (§7.2 gradient-boosting nonlinearity check, then §7.3–7.4 ridge + PCR/PLS).

**Run top-to-bottom.** Each section is a self-contained script that defines its own helpers (`load_data`, `dm_tstat`, `summarize`, …); the sections deliberately reuse those names, so each redefines and runs before the next one redefines them. Running the whole notebook in order reproduces the paper; re-running an earlier section's final cell after a later section has loaded will pick up the later section's helpers.

Requires `NG_m_final.csv` in the working directory, plus `pandas numpy statsmodels scipy scikit-learn arch matplotlib`.

Every section now runs for real — no stubs. All sections are the supplied source verbatim, with one edit: the §7.2 nonlinearity script's data path was changed from `NG_m_final_full.csv` to `NG_m_final.csv` so the whole notebook runs on one dataset (revert that one line if you need the `_full` file).

In [1]:
# Run this first -- keeps output clean for GitHub (silences non-fatal library warnings:
# e.g. statsmodels KPSS InterpolationWarning, sklearn ConvergenceWarning, arch scaling notes).
import warnings
warnings.filterwarnings("ignore")
try:
    import statsmodels.api as _sm
    from statsmodels.tools.sm_exceptions import InterpolationWarning, ConvergenceWarning, ValueWarning
    for _w in (InterpolationWarning, ConvergenceWarning, ValueWarning):
        warnings.simplefilter("ignore", _w)
except Exception:
    pass

---

## Section 3.3 — Gregory–Hansen cointegration test (endogenous break)

Source: `ttf_gregory_hansen.py`

<details><summary>original module docstring</summary>

```
ttf_gregory_hansen.py

Gregory-Hansen (1996) test for cointegration among log TTF, log JKM, log HH
with a SINGLE structural break at an UNKNOWN date. Where the Engle-Granger and
PSS bounds tests in the paper assume a fixed (economically-imposed) 2021-09
break, this lets the DATA choose the break location.

Method (Gregory & Hansen 1996, J. Econometrics 70:99-126):
  For each candidate break fraction tau in a trimmed window (default 15%-85%):
    build a level dummy D_t = 1{t > tau}; fit the break-augmented cointegrating
    regression by OLS; take the ADF t-statistic on the residuals.
  The GH statistic is ADF* = min over tau of that ADF t-stat (most negative =
  strongest evidence of a stationary equilibrium error somewhere), and the
  ESTIMATED BREAK DATE is the tau that attains it.

Two model variants:
  - Model C  (level shift):   log_TTF = mu1 + mu2*D + b1*log_JKM + b2*log_HH + e
  - Model C/S (regime shift): also interacts the slopes with D (full break in
    intercept AND cointegrating vector).

A more negative ADF* than the GH critical value rejects "no cointegration";
either way the argmin gives the most likely break date. (GH is single-break; for
multiple breaks e.g. a 2021 onset AND a 2023 normalisation, a Bai-Perron test
would be the extension.)

Requirements: statsmodels, numpy, pandas
```

</details>

In [2]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller

DATA_PATH = "NG_m_final.csv"
SAMPLE_START = "2015-01-01"
TRIM = 0.15                      # exclude first/last 15% as candidate breaks
LONGRUN_X = ["log_JKM", "log_HH"]   # m = 2 regressors

# Gregory-Hansen (1996) Table 1 asymptotic critical values for ADF*/Zt*, m=2.
# (Reference values -- confirm against the original table before citing.)
GH_CV = {
    "C":  {"1%": -5.44, "5%": -4.92, "10%": -4.69},   # level shift
    "CS": {"1%": -5.97, "5%": -5.50, "10%": -5.23},   # regime shift
}


def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    jkm = df["JKM(USD/mmbtu)"].astype(float).copy()
    jkm[np.isclose(jkm, 7.86)] = np.nan     # replace the 4-month 2017 fill
    df["JKM(USD/mmbtu)"] = np.exp(np.log(jkm).interpolate(limit_direction="both"))
    df["log_TTF"] = np.log(df["TTF(USD/mmbtu)"])
    df["log_JKM"] = np.log(df["JKM(USD/mmbtu)"])
    df["log_HH"] = np.log(df["HH(USD/mmbtu)"])
    return df.loc[df["Date"] >= SAMPLE_START].reset_index(drop=True)


def adf_resid(resid):
    """ADF t-statistic on residuals, no deterministic term (already de-meaned)."""
    try:
        return adfuller(resid, regression="n", autolag="AIC")[0]
    except (TypeError, ValueError):
        return adfuller(resid, regression="nc", autolag="AIC")[0]   # older statsmodels


def gh_test(y, X, model="C", trim=TRIM):
    n = len(y)
    lo, hi = int(np.floor(n * trim)), int(np.ceil(n * (1 - trim)))
    results = []
    for tau in range(lo, hi):
        D = (np.arange(n) > tau).astype(float)
        cols = [np.ones(n), D] + [X[:, j] for j in range(X.shape[1])]
        if model == "CS":
            cols += [D * X[:, j] for j in range(X.shape[1])]
        Z = np.column_stack(cols)
        beta, *_ = np.linalg.lstsq(Z, y, rcond=None)
        resid = y - Z @ beta
        results.append((tau, adf_resid(resid)))
    res = pd.DataFrame(results, columns=["tau", "adf"])
    star = res.loc[res["adf"].idxmin()]
    return res, int(star["tau"]), float(star["adf"])


def main():
    df = load_data()
    y = df["log_TTF"].values
    X = df[LONGRUN_X].values
    dates = df["Date"].values
    print("Gregory-Hansen cointegration test with unknown break")
    print("log_TTF ~ log_JKM + log_HH   |   sample %s..%s  (n=%d, m=2 regressors)\n"
          % (df["Date"].min().date(), df["Date"].max().date(), len(df)))

    for model, name in [("C", "Model C  (level shift)"),
                        ("CS", "Model C/S (regime shift: intercept + slopes)")]:
        res, tau, adfstar = gh_test(y, X, model=model)
        cv = GH_CV[model]
        brk = pd.Timestamp(dates[tau]).date()
        verdict = ("reject no-cointegration at 1%" if adfstar < cv["1%"] else
                   "reject at 5%" if adfstar < cv["5%"] else
                   "reject at 10%" if adfstar < cv["10%"] else
                   "cannot reject no-cointegration")
        print("=" * 68)
        print(name)
        print("=" * 68)
        print("  ADF* (min over break dates) = %.3f" % adfstar)
        print("  estimated break date        = %s  (obs %d of %d)" % (brk, tau + 1, len(df)))
        print("  GH crit. values (m=2): 1%% %.2f  5%% %.2f  10%% %.2f  -> %s"
              % (cv["1%"], cv["5%"], cv["10%"], verdict))
        top = res.nsmallest(5, "adf")
        print("  most likely break dates (5 lowest ADF):")
        for _, r in top.iterrows():
            print("     %s   ADF=%.3f" % (pd.Timestamp(dates[int(r['tau'])]).date(), r["adf"]))
        print()

    print("NOTE: GH estimates a SINGLE break. If both a 2021 onset and a 2023")
    print("normalisation are of interest, a Bai-Perron multiple-break test is the extension.")
    print("Critical values are Gregory-Hansen (1996) Table 1, m=2 -- verify before citing.")

In [3]:
# ---- run this section ----
main()

Gregory-Hansen cointegration test with unknown break
log_TTF ~ log_JKM + log_HH   |   sample 2015-01-01..2026-05-01  (n=137, m=2 regressors)

Model C  (level shift)
  ADF* (min over break dates) = -6.683
  estimated break date        = 2021-09-01  (obs 81 of 137)
  GH crit. values (m=2): 1% -5.44  5% -4.92  10% -4.69  -> reject no-cointegration at 1%
  most likely break dates (5 lowest ADF):
     2021-09-01   ADF=-6.683
     2021-04-01   ADF=-6.557
     2022-01-01   ADF=-6.527
     2022-02-01   ADF=-6.513
     2021-05-01   ADF=-6.505

Model C/S (regime shift: intercept + slopes)
  ADF* (min over break dates) = -6.979
  estimated break date        = 2021-09-01  (obs 81 of 137)
  GH crit. values (m=2): 1% -5.97  5% -5.50  10% -5.23  -> reject no-cointegration at 1%
  most likely break dates (5 lowest ADF):
     2021-09-01   ADF=-6.979
     2022-01-01   ADF=-6.806
     2021-04-01   ADF=-6.769
     2021-08-01   ADF=-6.743
     2022-02-01   ADF=-6.721

NOTE: GH estimates a SINGLE break. If 

---

## Section 4 (baseline) — Core model

Source: `ttf_model.py`

<details><summary>original module docstring</summary>

```
ttf_model.py

An econometric model for forecasting month-ahead TTF (European natural
gas) prices from European supply, demand, and weather fundamentals.

Pipeline:
  1. Load monthly European gas market data.
  2. Engineer deseasonalized ("anomaly") predictors: momentum, storage
     change, and European heating-degree-days -- each compared only to its
     own expanding, prior-years-only calendar-month average, so no
     predictor ever uses future information.
  3. Stationarity-check every candidate series (ADF + KPSS). An LNG
     sendout variable was tested during development and excluded: both
     tests agreed it (and its deseasonalized anomaly) were non-stationary,
     and a stationary growth-rate version of it turned out to be
     statistically insignificant. It is kept out of the final model here;
     see `check_stationarity()` / `stationarity_report()` below if you
     want to re-run that check yourself, or re-add a revised LNG variable.
  4. Fit OLS with Newey-West HAC standard errors.
  5. Run regression diagnostics: Durbin-Watson, residual autocorrelation,
     Breusch-Pagan heteroskedasticity test.
  6. Backtest out-of-sample with an expanding-window walk-forward loop,
     benchmarked against a random walk, a momentum-only model, and the
     historical mean.
  7. Repeat steps 2-6 for 1-, 2-, and 3-month-ahead horizons.
  8. Produce the two summary charts used in the accompanying report.

Requirements: pandas, numpy, statsmodels, matplotlib, scipy
```

</details>

In [4]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

DATA_PATH = "NG_m_final.csv"
TRAIN_END = "2025-01-01"        # training window: everything before this date
MIN_TRAIN_OBS = 60              # minimum months of history before the walk-forward backtest starts

In [5]:
# --------------------------------------------------------------------------- #
# 1. Data loading and feature engineering
# --------------------------------------------------------------------------- #

def load_data(path: str = DATA_PATH) -> pd.DataFrame:
    """Load the raw dataset and add the base date/month columns."""
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df["month"] = df["Date"].dt.month
    df["TTF"] = df["TTF(USD/mmbtu)"]
    return df


def expanding_seasonal_mean(series: pd.Series, month: pd.Series) -> pd.Series:
    """
    Out-of-sample-safe calendar-month climatology.

    For each row, returns the mean of all PRIOR years' values for the same
    calendar month (the current observation is excluded). This is what makes
    the resulting anomaly safe to use as a predictor: at any point in time,
    it only reflects information that was actually available up to that
    point, never future data.
    """
    out = pd.Series(index=series.index, dtype=float)
    tmp = pd.DataFrame({"val": series, "month": month})
    for _, grp in tmp.groupby("month"):
        clim = grp["val"].expanding().mean().shift(1)
        out.loc[grp.index] = clim
    return out


def build_features(df: pd.DataFrame, horizon: int = 1) -> pd.DataFrame:
    """
    Build the target and the three deseasonalized, horizon-lagged predictors
    for a given forecast horizon H (in months).

    Target:      H-month forward log return of TTF, dlogH_TTF(t) = log(TTF_t) - log(TTF_{t-H})
    Predictors (all anomalies, lagged H months so they are predetermined
    relative to the return they forecast):
      - momentum         : deseasonalized H-month TTF return itself
      - storage_change   : deseasonalized month-over-month change in EU+UK storage (bcm)
      - HDD               : deseasonalized European heating-degree-days

    Note: an LNG sendout variable was deliberately left out -- see module
    docstring and check_stationarity() below.
    """
    out = df[["Date", "month", "TTF"]].copy()

    dlogH = np.log(df["TTF"]).diff(horizon)
    dlogH_anom = dlogH - expanding_seasonal_mean(dlogH, df["month"])
    out["y"] = dlogH
    out["momentum"] = dlogH_anom.shift(horizon)

    storage_change = df["EU+UK_av_storage(bcm)"].diff()
    out["storage_change_anom"] = (
        storage_change - expanding_seasonal_mean(storage_change, df["month"])
    ).shift(horizon)

    out["HDD_anom"] = (
        df["Europe_HDD"] - expanding_seasonal_mean(df["Europe_HDD"], df["month"])
    ).shift(horizon)

    return out


FEATURE_COLS = ["momentum", "storage_change_anom", "HDD_anom"]

In [6]:
# --------------------------------------------------------------------------- #
# 2. Stationarity checks
# --------------------------------------------------------------------------- #

def check_stationarity(series: pd.Series, name: str = "") -> dict:
    """
    Run ADF (H0: unit root) and KPSS (H0: stationary) on a series and
    return both test statistics plus a simple combined verdict.

    A series is flagged 'stationary' only if ADF rejects its null AND KPSS
    fails to reject its null (i.e. both tests agree). Anything else is
    flagged for manual review -- as with storage_change_anom in this model,
    a conflicting verdict is not automatically disqualifying, especially
    for a variable that is economically implausible as a genuine unit-root
    process (e.g. the change in a physically bounded stock).
    """
    s = series.dropna()
    adf_stat, adf_p, *_ = adfuller(s, autolag="AIC")
    kpss_stat, kpss_p, *_ = kpss(s, regression="c", nlags="auto")

    adf_reject_unit_root = adf_p < 0.05
    kpss_fail_reject_stationary = kpss_p > 0.05
    verdict = (
        "stationary" if (adf_reject_unit_root and kpss_fail_reject_stationary)
        else "non-stationary" if (not adf_reject_unit_root and not kpss_fail_reject_stationary)
        else "conflicting -- review manually"
    )
    return {
        "name": name, "adf_stat": adf_stat, "adf_p": adf_p,
        "kpss_stat": kpss_stat, "kpss_p": kpss_p, "verdict": verdict,
    }


def stationarity_report(df: pd.DataFrame, train_end: str = TRAIN_END) -> pd.DataFrame:
    """Run check_stationarity() over every series considered for this model."""
    train = df.loc[df["Date"] < train_end].copy()

    log_lng = np.log(train["EU+UK LNG_sendout(bcm)"])
    lng_anom = log_lng - expanding_seasonal_mean(log_lng, train["month"])
    dlog_lng = log_lng.diff()
    dlog_lng_anom = dlog_lng - expanding_seasonal_mean(dlog_lng, train["month"])

    dlog_ttf = np.log(train["TTF"]).diff()
    dlog_ttf_anom = dlog_ttf - expanding_seasonal_mean(dlog_ttf, train["month"])

    storage_change = train["EU+UK_av_storage(bcm)"].diff()
    storage_change_anom = storage_change - expanding_seasonal_mean(storage_change, train["month"])

    hdd_anom = train["Europe_HDD"] - expanding_seasonal_mean(train["Europe_HDD"], train["month"])

    series_to_check = {
        "dlog(TTF)": dlog_ttf,
        "momentum anomaly": dlog_ttf_anom,
        "storage_change_anom": storage_change_anom,
        "HDD_anom": hdd_anom,
        "log_LNG (level, excluded)": log_lng,
        "LNG_anom (excluded)": lng_anom,
        "dlog_LNG_anom (stationary but insignificant, excluded)": dlog_lng_anom,
    }
    return pd.DataFrame([check_stationarity(s, name) for name, s in series_to_check.items()])

In [7]:
# --------------------------------------------------------------------------- #
# 3. Estimation and diagnostics
# --------------------------------------------------------------------------- #

def fit_hac_model(feat: pd.DataFrame, train_end: str = TRAIN_END, extra_hac_lags: int = 0):
    """Fit OLS with Newey-West HAC standard errors on the training window."""
    train = feat.loc[feat["Date"] < train_end, ["Date", "y"] + FEATURE_COLS].dropna()
    y = train["y"]
    X = sm.add_constant(train[FEATURE_COLS])

    n = len(train)
    maxlags = int(np.floor(4 * (n / 100) ** (2 / 9))) + extra_hac_lags
    model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags, "use_correction": True})
    return model, train


def run_diagnostics(model, train: pd.DataFrame) -> dict:
    """Durbin-Watson, residual lag-1 autocorrelation, and Breusch-Pagan test."""
    resid = model.resid
    dw = durbin_watson(resid)
    resid_ar1 = np.corrcoef(resid[:-1], resid[1:])[0, 1]

    X = sm.add_constant(train[FEATURE_COLS])
    bp_lm, bp_p, _, _ = het_breuschpagan(resid, X)

    return {
        "durbin_watson": dw,
        "resid_ar1": resid_ar1,
        "breusch_pagan_lm": bp_lm,
        "breusch_pagan_p": bp_p,
    }

In [8]:
# --------------------------------------------------------------------------- #
# 4. Walk-forward out-of-sample backtest
# --------------------------------------------------------------------------- #

def walk_forward_backtest(feat: pd.DataFrame, min_train: int = MIN_TRAIN_OBS) -> pd.DataFrame:
    """
    Expanding-window, one-step-ahead backtest.

    At each step, refit the full model (and a momentum-only benchmark) using
    only data available up to that point, then forecast the next observation.
    Also records a random-walk (zero-change) and historical-mean benchmark.
    """
    data = feat[["Date", "y"] + FEATURE_COLS].dropna().reset_index(drop=True)
    rows = []

    for i in range(min_train, len(data)):
        train, test = data.iloc[:i], data.iloc[i]

        X_train = sm.add_constant(train[FEATURE_COLS])
        full_fit = sm.OLS(train["y"], X_train).fit()
        X_test = sm.add_constant(test[FEATURE_COLS].to_frame().T, has_constant="add")
        pred_full = full_fit.predict(X_test).iloc[0]

        X_train_mom = sm.add_constant(train[["momentum"]])
        mom_fit = sm.OLS(train["y"], X_train_mom).fit()
        X_test_mom = sm.add_constant(test[["momentum"]].to_frame().T, has_constant="add")
        pred_mom = mom_fit.predict(X_test_mom).iloc[0]

        rows.append({
            "Date": test["Date"],
            "actual": test["y"],
            "pred_full": pred_full,
            "pred_momentum": pred_mom,
            "pred_random_walk": 0.0,
            "pred_hist_mean": train["y"].mean(),
        })

    return pd.DataFrame(rows)


def backtest_summary(res: pd.DataFrame) -> pd.DataFrame:
    """RMSE, MAE, directional hit rate, and OOS R^2 vs. random walk for each model."""
    rows = []
    rw_err = res["actual"] - res["pred_random_walk"]
    for label, col in [
        ("Full model", "pred_full"),
        ("Momentum-only", "pred_momentum"),
        ("Random walk", "pred_random_walk"),
        ("Historical mean", "pred_hist_mean"),
    ]:
        err = res["actual"] - res[col]
        rmse = np.sqrt((err ** 2).mean())
        mae = err.abs().mean()
        hit = np.nan if label == "Random walk" else (np.sign(res["actual"]) == np.sign(res[col])).mean()
        oos_r2 = 1 - (err ** 2).sum() / (rw_err ** 2).sum() if label != "Random walk" else np.nan
        rows.append({"Model": label, "RMSE": rmse, "MAE": mae, "Hit rate": hit, "OOS R2 vs RW": oos_r2})

    summary = pd.DataFrame(rows)

    full_err = res["actual"] - res["pred_full"]
    loss_diff = rw_err ** 2 - full_err ** 2
    t_stat = loss_diff.mean() / (loss_diff.std(ddof=1) / np.sqrt(len(loss_diff)))
    summary.attrs["loss_diff_tstat_full_vs_rw"] = t_stat
    return summary

In [9]:
# --------------------------------------------------------------------------- #
# 5. Plots
# --------------------------------------------------------------------------- #

def plot_actual_vs_fitted(df: pd.DataFrame, feat: pd.DataFrame, model, train_end: str = TRAIN_END,
                           outpath: str = "actual_vs_fitted.png"):
    """Reconstruct the fitted price level and plot it against the actual price."""
    full = feat[["Date", "TTF"] + FEATURE_COLS].dropna().reset_index(drop=True)
    X_full = sm.add_constant(full[FEATURE_COLS], has_constant="add")
    full["pred_dlog"] = model.predict(X_full)

    prev_price = df[["Date", "TTF"]].rename(columns={"TTF": "TTF_prev"})
    prev_price = prev_price.assign(Date=prev_price["Date"] + pd.offsets.MonthBegin(1))
    full = full.merge(prev_price, on="Date", how="left")
    full["fitted_TTF"] = full["TTF_prev"] * np.exp(full["pred_dlog"])

    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.plot(full["Date"], full["TTF"], label="Actual TTF", color="#1f4e79", linewidth=1.8)
    ax.plot(full["Date"], full["fitted_TTF"], label="Fitted TTF", color="#c0392b",
            linewidth=1.6, linestyle="--")
    split = pd.Timestamp(train_end)
    ax.axvline(split, color="grey", linewidth=1, linestyle=":")
    ax.text(split, ax.get_ylim()[1] * 0.97, "  in-sample | out-of-sample", fontsize=8.5, color="grey", va="top")
    ax.set_title("Actual vs. Fitted TTF Price ($/mmbtu)", fontsize=12)
    ax.set_ylabel("TTF (USD/mmbtu)")
    ax.legend(frameon=False, loc="upper left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close(fig)
    return full


def plot_model_adjustment(full: pd.DataFrame, train_end: str = TRAIN_END,
                           outpath: str = "model_adjustment.png"):
    """Isolate and plot the model's adjustment over a naive random-walk forecast."""
    full = full.copy()
    full["model_adjustment"] = full["fitted_TTF"] - full["TTF_prev"]
    full["actual_change"] = full["TTF"] - full["TTF_prev"]

    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.plot(full["Date"], full["actual_change"], label="Actual month-over-month change",
            color="#1f4e79", linewidth=1.6)
    ax.plot(full["Date"], full["model_adjustment"], label="Model's adjustment vs. naive forecast",
            color="#c0392b", linewidth=1.6, linestyle="--")
    ax.axhline(0, color="grey", linewidth=0.8)
    split = pd.Timestamp(train_end)
    ax.axvline(split, color="grey", linewidth=1, linestyle=":")
    ax.set_title("Model's Price Adjustment vs. Actual Price Change (USD/mmbtu)", fontsize=12)
    ax.set_ylabel("USD/mmbtu")
    ax.legend(frameon=False, loc="upper left")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close(fig)

In [10]:
# --------------------------------------------------------------------------- #
# 6. Main
# --------------------------------------------------------------------------- #

def main():
    df = load_data()

    print("=" * 70)
    print("Stationarity checks (training sample)")
    print("=" * 70)
    print(stationarity_report(df).to_string(index=False))

    print("\n" + "=" * 70)
    print("Horizon comparison (H = 1, 2, 3 months ahead)")
    print("=" * 70)
    for H in (1, 2, 3):
        feat = build_features(df, horizon=H)
        model, train = fit_hac_model(feat, extra_hac_lags=H - 1)
        diag = run_diagnostics(model, train)
        res = walk_forward_backtest(feat)
        summary = backtest_summary(res)

        print(f"\n--- H = {H} month(s) ---")
        print(model.summary())
        print(f"Durbin-Watson: {diag['durbin_watson']:.3f}   "
              f"Residual AR(1): {diag['resid_ar1']:.3f}   "
              f"Breusch-Pagan LM: {diag['breusch_pagan_lm']:.2f} (p={diag['breusch_pagan_p']:.3f})")
        print(summary.to_string(index=False))
        print(f"Loss-differential t-stat (full model vs. random walk): "
              f"{summary.attrs['loss_diff_tstat_full_vs_rw']:.2f}")

        if H == 1:
            full = plot_actual_vs_fitted(df, feat, model)
            plot_model_adjustment(full)
            print("\nSaved actual_vs_fitted.png and model_adjustment.png")

In [11]:
# ---- run this section ----
main()

Stationarity checks (training sample)
                                                  name   adf_stat        adf_p  kpss_stat   kpss_p        verdict
                                             dlog(TTF)  -7.813048 6.986648e-12   0.083409 0.100000     stationary
                                      momentum anomaly  -8.078444 1.483583e-12   0.144206 0.100000     stationary
                                   storage_change_anom  -5.569164 1.480953e-06   0.071317 0.100000     stationary
                                              HDD_anom  -5.046797 1.791651e-05   0.238115 0.100000     stationary
                             log_LNG (level, excluded)  -1.594699 4.862561e-01   1.445546 0.010000 non-stationary
                                   LNG_anom (excluded)  -2.493277 1.171186e-01   0.654273 0.017702 non-stationary
dlog_LNG_anom (stationary but insignificant, excluded) -12.236372 1.025938e-22   0.067402 0.100000     stationary

Horizon comparison (H = 1, 2, 3 months ahead)

--

---

## Section 4 — European gas modelling (core / Conditional I & II / lean)

Source: `European_monthly_gas_model.py`

<details><summary>original module docstring</summary>

```
European_monthly_gas_model.py

SINGLE entry point for the full self-contained European month-ahead (H=1) TTF
study. This one file merges what were previously four separate scripts and runs
all four sub-models in sequence from one `main()`:

  1. DEPLOYABLE        (was ttf_european_model.py)
       Every predictor lagged one month -> an honest, real-world forecast.
       Full machinery: stationarity gate, univariate HAC screen, collinearity
       pruning, training-only selection, walk-forward, Diebold-Mariano, and a
       regime split with a joint Wald test for regime dependence.

  2. CONDITIONAL (all fundamentals)  (was ttf_european_conditional.py)
       Perfect foresight of EVERY same-month fundamental (contemporaneous),
       data-driven selection. An explanatory CEILING, not a deployable forecast.

  3. EXOGENOUS CEILING  (was ttf_european_exogenous.py)
       Pre-specified, endogeneity-free ceiling: only fundamentals price cannot
       cause (weather) plus semi-endogenous storage, all with perfect foresight.

  4. LEAN               (was ttf_european_exogenous_conditional2.py / ttf_european_lean.py)
       A deliberately lean three-parameter perfect-foresight robustness check,
       to separate "signal is genuinely weak" from "ceiling was over-parameterized".

Common design (shared by all four):
  * Every predictor is a deseasonalized anomaly vs an EXPANDING, prior-years-only
    calendar-month climatology, so nothing uses future information.
  * Target y = one-month log return of front-month TTF (USD/mmbtu). H = 1.
  * momentum is always LAGGED (predetermined); using the contemporaneous return
    would be circular. In the deployable model ALL predictors are lagged; in the
    conditional/exogenous/lean models the fundamentals enter contemporaneously
    (that is the intended, and only, look-ahead).
  * Selection and in-sample fit use data strictly before TRAIN_END; evaluation is
    an expanding walk-forward with a Newey-West Diebold-Mariano test vs a random
    walk, reported over the full backtest and the clean post-TRAIN_END slice.

European / EU+UK fundamentals only (no JKM / Henry Hub / global LNG / financials).
Toy / educational model, not a trading or investment tool.

Run:  python European_monthly_gas_model.py   (runs all four sub-models)

Requirements: pandas, numpy, statsmodels, scipy
```

</details>

In [12]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss

In [13]:
# --------------------------------------------------------------------------- #
# Shared configuration
# --------------------------------------------------------------------------- #

DATA_PATH = "NG_m_final.csv"
TRAIN_END = "2025-01-01"     # selection & in-sample fit use data strictly before this
MIN_TRAIN_OBS = 60           # months of history before the walk-forward begins
HORIZON = 1                  # months ahead (H=1 only, per the current experiment)

N_SELECT = 6                 # max predictors in a selected model (incl. momentum)
CORR_PRUNE_THRESH = 0.70     # drop the less-significant of any candidate pair above this |corr|
SCREEN_P_MAX = 0.10          # a candidate must clear this univariate HAC p-value to be selectable

# Calendar crisis window (acute run-up through post-peak normalization).
CRISIS_START = "2021-09-01"
CRISIS_END = "2023-07-01"    # crisis regime = [START, END); everything else = calm

# The always-excluded global block, kept here only for documentation / assertion.
GLOBAL_COLS_EXCLUDED = [
    "JKM(USD/mmbtu)", "HH(USD/mmbtu)", "USD-EUR_FX", "VIX",
    "US_GWDD", "NE_Asia_GWDD", "Atlantic_ACE", "Gulf_storm_days",
    "US_GWDD_anomaly", "NE_Asia_GWDD_anomaly", "Atlantic_ACE_anomaly",
    "Global LNG nameplate capacity", "Global LNG capacity offline",
    "CH+JP+KR LNG imports", "EG LNG imports", "IN LNG imports",
    "QA+AU+US LNG exports", "ID+MY+BN LNG exports", "NG LNG exports",
]

In [14]:
# --------------------------------------------------------------------------- #
# Shared: data + no-look-ahead anomaly builders
# --------------------------------------------------------------------------- #

def load_data(path: str = DATA_PATH) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df["month"] = df["Date"].dt.month
    df["TTF"] = df["TTF(USD/mmbtu)"]
    return df


def expanding_seasonal_mean(series: pd.Series, month: pd.Series) -> pd.Series:
    """Mean of all PRIOR years' values for the same calendar month (current excluded)."""
    out = pd.Series(index=series.index, dtype=float)
    tmp = pd.DataFrame({"val": series, "month": month})
    for _, grp in tmp.groupby("month"):
        out.loc[grp.index] = grp["val"].expanding().mean().shift(1)
    return out


def level_anom(df, col):
    return df[col] - expanding_seasonal_mean(df[col], df["month"])


def change_anom(df, col):
    ch = df[col].diff()
    return ch - expanding_seasonal_mean(ch, df["month"])


def dlog_anom(df, col):
    dl = np.log(df[col]).diff()
    return dl - expanding_seasonal_mean(dl, df["month"])


TRANSFORMS = {"level_anom": level_anom, "change_anom": change_anom, "dlog_anom": dlog_anom}


def hac_lags(n: int, extra: int = 0) -> int:
    return int(np.floor(4 * (n / 100) ** (2 / 9))) + extra

In [15]:
# --------------------------------------------------------------------------- #
# Shared: fit / walk-forward / Diebold-Mariano / summarize / R2
# --------------------------------------------------------------------------- #

def fit_hac(feat: pd.DataFrame, cols, train_end: str = TRAIN_END):
    """HAC (Newey-West) OLS on the training window; returns the fitted model."""
    d = feat.loc[feat["Date"] < train_end, ["y"] + cols].dropna()
    return sm.OLS(d["y"], sm.add_constant(d[cols])).fit(
        cov_type="HAC", cov_kwds={"maxlags": hac_lags(len(d)), "use_correction": True})


def r2(feat: pd.DataFrame, cols, train_end: str = TRAIN_END):
    """Plain in-sample R2 (and n) of an OLS fit on the training window."""
    d = feat.loc[feat["Date"] < train_end, ["y"] + cols].dropna()
    return sm.OLS(d["y"], sm.add_constant(d[cols])).fit().rsquared, len(d)


def walk_forward(feat: pd.DataFrame, model_cols: dict, min_train: int = MIN_TRAIN_OBS) -> pd.DataFrame:
    """
    Expanding-window one-step-ahead backtest for several models at once.
    `model_cols` maps model name -> list of feature columns. Coefficients are
    refit each step (the feature SET is fixed, chosen on the training window
    only). Random walk (0) and historical mean are added automatically.
    """
    all_cols = sorted({c for cols in model_cols.values() for c in cols})
    data = feat[["Date", "y"] + all_cols].dropna().reset_index(drop=True)
    rows = []
    for i in range(min_train, len(data)):
        train, test = data.iloc[:i], data.iloc[i]
        rec = {"Date": test["Date"], "actual": test["y"],
               "pred_random_walk": 0.0, "pred_hist_mean": train["y"].mean()}
        for name, cols in model_cols.items():
            fit = sm.OLS(train["y"], sm.add_constant(train[cols])).fit()
            Xte = sm.add_constant(test[cols].to_frame().T, has_constant="add")
            rec[f"pred_{name}"] = fit.predict(Xte).iloc[0]
        rows.append(rec)
    return pd.DataFrame(rows)

In [16]:
def dm_tstat(actual, pred_worse, pred_better, h: int = HORIZON) -> float:
    """
    Diebold-Mariano statistic for squared-error loss with a Newey-West variance
    (h-1 lags). Positive => `pred_better` has significantly lower MSE.
    """
    a = np.asarray(actual, float)
    d = (a - np.asarray(pred_worse, float)) ** 2 - (a - np.asarray(pred_better, float)) ** 2
    d = d[~np.isnan(d)]
    n = len(d)
    dbar = d.mean()
    lag = max(h - 1, 0)
    var = ((d - dbar) ** 2).mean()
    for l in range(1, lag + 1):
        cov = ((d[l:] - dbar) * (d[:-l] - dbar)).mean()
        var += 2 * (1 - l / (lag + 1)) * cov
    return dbar / np.sqrt(var / n) if var > 0 else np.nan


def summarize(res: pd.DataFrame, model_names) -> pd.DataFrame:
    rw = res["pred_random_walk"]
    rw_err = res["actual"] - rw
    rows = []
    for name in model_names + ["random_walk", "hist_mean"]:
        pred = res[f"pred_{name}"]
        err = res["actual"] - pred
        rows.append({"Model": name, "RMSE": np.sqrt((err ** 2).mean()),
                     "Hit rate": np.nan if name == "random_walk" else (np.sign(res["actual"]) == np.sign(pred)).mean(),
                     "OOS R2 vs RW": np.nan if name == "random_walk" else 1 - (err ** 2).sum() / (rw_err ** 2).sum(),
                     "DM t vs RW": np.nan if name == "random_walk" else dm_tstat(res["actual"], rw, pred),
                     "n": len(res)})
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------- #
# Shared: stationarity gate + univariate screen + collinearity prune + select
# (used by the DEPLOYABLE and CONDITIONAL-all sub-models)
# --------------------------------------------------------------------------- #

In [17]:
def check_stationarity(series: pd.Series, name: str = "") -> dict:
    s = series.dropna()
    adf_p = adfuller(s, autolag="AIC")[1]
    kpss_p = kpss(s, regression="c", nlags="auto")[1]
    stationary = (adf_p < 0.05) and (kpss_p > 0.05)
    borderline = (adf_p < 0.05) or (kpss_p > 0.05)   # at least one test says stationary
    verdict = "stationary" if stationary else ("conflicting" if borderline else "non-stationary")
    return {"name": name, "adf_p": adf_p, "kpss_p": kpss_p, "verdict": verdict, "usable": borderline}


def stationarity_gate(feat: pd.DataFrame, names, train_end: str = TRAIN_END):
    train = feat.loc[feat["Date"] < train_end]
    rep = pd.DataFrame([check_stationarity(train[nm], nm) for nm in names])
    return rep, rep.loc[rep["usable"], "name"].tolist()


def univariate_screen(feat: pd.DataFrame, names, meta: dict, train_end: str = TRAIN_END) -> pd.DataFrame:
    """One-predictor HAC regressions on the training window; ranked by |t|."""
    train = feat.loc[feat["Date"] < train_end]
    rows = []
    for nm in names:
        d = train[["y", nm]].dropna()
        if len(d) < 30:
            continue
        m = sm.OLS(d["y"], sm.add_constant(d[[nm]])).fit(
            cov_type="HAC", cov_kwds={"maxlags": hac_lags(len(d)), "use_correction": True})
        rows.append({"name": nm, "coef": m.params[nm], "t": m.tvalues[nm], "p": m.pvalues[nm],
                     "R2": m.rsquared, "exp_sign": meta[nm]["sign"],
                     "sign_ok": np.sign(m.params[nm]) == meta[nm]["sign"],
                     "n": len(d), "note": meta[nm]["note"]})
    out = pd.DataFrame(rows)
    return out.reindex(out["t"].abs().sort_values(ascending=False).index).reset_index(drop=True)

In [18]:
def prune_collinear(feat: pd.DataFrame, screen: pd.DataFrame, train_end: str = TRAIN_END,
                    thresh: float = CORR_PRUNE_THRESH):
    """Greedily drop the lower-|t| member of any candidate pair correlated above `thresh`."""
    train = feat.loc[feat["Date"] < train_end]
    ranked = screen["name"].tolist()               # already sorted by |t| desc
    corr = train[ranked].corr().abs()
    kept, dropped = [], {}
    for nm in ranked:
        clash = next((k for k in kept if corr.loc[nm, k] > thresh), None)
        if clash is None:
            kept.append(nm)
        else:
            dropped[nm] = clash
    return kept, dropped


def select_model(screen: pd.DataFrame, kept_pool, n_select: int = N_SELECT,
                 p_max: float = SCREEN_P_MAX):
    """
    Compact model: momentum as anchor, then the strongest surviving candidates
    (by |t|) that clear p_max, up to n_select total. Training-window stats only.
    """
    ordered = [nm for nm in screen["name"].tolist() if nm in kept_pool]
    sig = set(screen.loc[screen["p"] < p_max, "name"])
    chosen = ["momentum"] if "momentum" in kept_pool else []
    for nm in ordered:
        if len(chosen) >= n_select:
            break
        if nm == "momentum" or nm not in sig:
            continue
        chosen.append(nm)
    return chosen

In [19]:
# --------------------------------------------------------------------------- #
# Shared: regime flags + regime diagnostics
# --------------------------------------------------------------------------- #

def add_regime(feat: pd.DataFrame) -> pd.DataFrame:
    """
    Add two regime flags:
      crisis  -- calendar dummy for the 2021-22 energy crisis window.
      highvol -- LOOK-AHEAD-SAFE: trailing 6-month realized vol of y, shifted 1,
                 above its own expanding median (also shifted).
    """
    f = feat.copy()
    f["crisis"] = ((f["Date"] >= CRISIS_START) & (f["Date"] < CRISIS_END)).astype(int)
    vol = f["y"].rolling(6).std().shift(1)
    med = vol.expanding().median().shift(1)
    f["highvol"] = (vol > med).astype(float)
    return f


def _subsample_fit(feat, cols, mask):
    d = feat.loc[mask, ["y"] + cols].dropna()
    if len(d) < len(cols) + 5:
        return None, d
    m = sm.OLS(d["y"], sm.add_constant(d[cols])).fit(
        cov_type="HAC", cov_kwds={"maxlags": hac_lags(len(d)), "use_correction": True})
    return m, d

In [20]:
def regime_subsample(feat, cols, regime_col, hi_label, lo_label):
    """Side-by-side HAC coefficients (coef, t) in the two regimes, plus n and R2."""
    tab = {c: {} for c in ["const"] + cols}
    meta = {}
    for val, lab in [(1, hi_label), (0, lo_label)]:
        m, d = _subsample_fit(feat, cols, feat[regime_col] == val)
        meta[lab] = {"n": len(d), "R2": (m.rsquared if m is not None else np.nan)}
        for c in ["const"] + cols:
            tab[c][f"coef[{lab}]"] = (m.params[c] if m is not None else np.nan)
            tab[c][f"t[{lab}]"] = (m.tvalues[c] if m is not None else np.nan)
    df = pd.DataFrame(tab).T[[f"coef[{hi_label}]", f"t[{hi_label}]",
                              f"coef[{lo_label}]", f"t[{lo_label}]"]]
    return df.round(4), meta


def regime_interaction(feat, cols, regime_col):
    """
    Pooled HAC regression y ~ const + regime + X + X:regime, with a joint Wald
    test that ALL interaction terms are zero (H0: coefficients don't differ by
    regime). A small joint p => coefficients are regime-dependent.
    """
    d = feat[["y", regime_col] + cols].dropna().copy()
    X = pd.DataFrame({"regime": d[regime_col].astype(float)}, index=d.index)
    inter = []
    for c in cols:
        X[c] = d[c].values
        ic = f"{c}_Xreg"
        X[ic] = (d[c] * d[regime_col]).values
        inter.append(ic)
    X = sm.add_constant(X)
    m = sm.OLS(d["y"], X).fit(cov_type="HAC",
                              cov_kwds={"maxlags": hac_lags(len(d)), "use_correction": True})
    names = list(X.columns)
    R = np.zeros((len(inter), len(names)))
    for i, ic in enumerate(inter):
        R[i, names.index(ic)] = 1.0
    w = m.wald_test(R, use_f=True)
    inter_t = {c: m.tvalues[f"{c}_Xreg"] for c in cols}
    return m, inter_t, float(np.squeeze(w.statistic)), float(np.squeeze(w.pvalue)), len(d)

In [21]:
def regime_report_full(feat, cols):
    """Rich regime split (coefficients + joint Wald) -- used for the deployable model."""
    feat = add_regime(feat)
    print("\n" + "=" * 78)
    print("Regime split -- selected European model: %s" % cols)
    print("=" * 78)
    for regime_col, hi, lo in [("crisis", "crisis", "calm"),
                               ("highvol", "high_vol", "low_vol")]:
        tag = ("Calendar crisis %s..%s" % (CRISIS_START[:7], CRISIS_END[:7])
               if regime_col == "crisis"
               else "Trailing-volatility (look-ahead-safe)")
        print("\n--- %s ---" % tag)
        sub, meta = regime_subsample(feat, cols, regime_col, hi, lo)
        for lab, mm in meta.items():
            print(f"  {lab:9}: n={mm['n']:3d}   R2={mm['R2']:.3f}")
        print(sub.to_string())
        m, inter_t, wstat, wp, npool = regime_interaction(feat, cols, regime_col)
        print("  interaction t-stats (coef differs in %s vs %s):" % (hi, lo))
        for c, t in inter_t.items():
            print(f"    {c:24} t = {t:+.2f}")
        print("  Joint Wald test (all interactions = 0):  F = %.2f,  p = %.3f   [n=%d]"
              % (wstat, wp, npool))
        print("  => %s" % ("coefficients ARE regime-dependent (signal concentrated in one regime)"
                           if wp < 0.10 else
                           "no significant regime dependence detected"))

In [22]:
def regime_report_simple(feat, cols):
    """Compact regime split (n and R2 only) -- used for the conditional-all model."""
    feat = add_regime(feat)
    print("\n" + "=" * 78)
    print("Regime split -- selected conditional model: %s" % cols)
    print("=" * 78)
    for rc, hi, lo in [("crisis", "crisis", "calm"), ("highvol", "high_vol", "low_vol")]:
        tag = ("Calendar crisis" if rc == "crisis" else "Trailing-volatility (look-ahead-safe)")
        print("\n--- %s ---" % tag)
        for val, lab in [(1, hi), (0, lo)]:
            m, _ = _subsample_fit(feat, cols, feat[rc] == val)
            print("  %-9s: %s" % (lab, "n<min" if m is None else "n=%d  R2=%.3f" % (int(m.nobs), m.rsquared)))


# --------------------------------------------------------------------------- #
# Candidate universe (shared by the DEPLOYABLE and CONDITIONAL-all sub-models).
# Expected sign is the hypothesized effect on the forward TTF return; it is used
# only for reporting, never to force selection.
# --------------------------------------------------------------------------- #

CANDIDATES = [
    ("momentum",              "TTF",                       "dlog_anom",  +1, "own 1-month log return (LAGGED)"),
    # --- EU+UK storage & gas balance ---
    ("storage_change_anom",   "EU+UK_av_storage(bcm)",     "change_anom", -1, "MoM storage change (build = bearish)"),
    ("storage_level_anom",    "EU+UK_av_storage(bcm)",     "level_anom",  -1, "storage level"),
    ("production_anom",       "EU+UK Production(bcm)",      "level_anom",  -1, "indigenous production"),
    ("net_piped_anom",        "EU+UK Net_piped(bcm)",      "level_anom",  -1, "net pipeline imports"),
    ("lng_imports_anom",      "EU+UK LNG imports",         "level_anom",  -1, "EU+UK LNG imports (supply)"),
    ("net_supply_anom",       "EU+UK Net_supply",          "level_anom",  -1, "total net supply"),
    ("total_demand_anom",     "EU+UK Total(bcm)",          "level_anom",  +1, "total gas demand"),
    ("nonpower_demand_anom",  "EU+UK Non_power(bcm)",       "level_anom",  +1, "non-power gas demand"),
    ("power_gas_demand_anom", "EU+UK Electricity(bcm)",    "level_anom",  +1, "gas-for-power demand (bcm)"),
    # --- EU+UK power generation mix (substitutes for gas in the stack) ---
    ("residual_load_anom",    "EU+UK Residual load",       "level_anom",  +1, "thermal power demand (load - wind/solar)"),
    ("coal_gen_anom",         "EU+UK Coal",                "level_anom",  -1, "coal generation (gas substitute)"),
    ("nuclear_gen_anom",      "EU+UK Nuclear",             "level_anom",  -1, "nuclear generation"),
    ("hydro_gen_anom",        "EU+UK Hydro_gen",           "level_anom",  -1, "hydro generation"),
    ("gas_burn_anom",         "EU+UK Fossil gas",          "level_anom",  +1, "gas-fired generation (demand; note: endogenous)"),
    # --- EU+UK supply / demand / generation as MONTH-OVER-MONTH CHANGES ---
    ("production_chg_anom",     "EU+UK Production(bcm)",   "change_anom", -1, "change in indigenous production"),
    ("net_piped_chg_anom",      "EU+UK Net_piped(bcm)",   "change_anom", -1, "change in net pipeline imports"),
    ("lng_imports_chg_anom",    "EU+UK LNG imports",      "change_anom", -1, "change in EU+UK LNG imports"),
    ("net_supply_chg_anom",     "EU+UK Net_supply",       "change_anom", -1, "change in total net supply"),
    ("total_demand_chg_anom",   "EU+UK Total(bcm)",       "change_anom", +1, "change in total gas demand"),
    ("nonpower_demand_chg_anom","EU+UK Non_power(bcm)",    "change_anom", +1, "change in non-power gas demand"),
    ("power_gas_demand_chg_anom","EU+UK Electricity(bcm)", "change_anom", +1, "change in gas-for-power demand (bcm)"),
    ("residual_load_chg_anom",  "EU+UK Residual load",    "change_anom", +1, "change in thermal power demand"),
    ("coal_gen_chg_anom",       "EU+UK Coal",             "change_anom", -1, "change in coal generation"),
    ("nuclear_gen_chg_anom",    "EU+UK Nuclear",          "change_anom", -1, "change in nuclear generation"),
    ("hydro_gen_chg_anom",      "EU+UK Hydro_gen",        "change_anom", -1, "change in hydro generation"),
    ("gas_burn_chg_anom",       "EU+UK Fossil gas",       "change_anom", +1, "change in gas-fired generation (endogenous)"),
    # --- Norway supply ---
    ("norway_prod_anom",      "Norway_gas_prod",           "level_anom",  -1, "Norwegian gas production"),
    ("norway_supplyred_anom", "Norway_supply_red",         "level_anom",  +1, "Norway supply reduction"),
    ("norway_planned_anom",   "Norway_planned_outage",     "level_anom",  +1, "Norway planned outages"),
    ("norway_unplanned_anom", "Norway_unplanned_outage",   "level_anom",  +1, "Norway unplanned outages"),
    # --- European weather ---
    ("hdd_anom",              "Europe_HDD",                "level_anom",  +1, "heating degree days"),
    ("cdd_anom",              "Europe_CDD",                "level_anom",  +1, "cooling degree days"),
    ("wind_anom",             "EU_wind_speed",             "level_anom",  -1, "wind speed (more wind = less gas)"),
    ("solar_anom",            "EU_solar",                  "level_anom",  -1, "solar irradiation"),
    ("precip_anom",           "Nordic_precip",             "level_anom",  -1, "Nordic precipitation (hydro inflows)"),
]

In [23]:
CANDIDATE_NAMES = [c[0] for c in CANDIDATES]
CANDIDATE_META = {c[0]: {"col": c[1], "transform": c[2], "sign": c[3], "note": c[4]} for c in CANDIDATES}

CORE_MODEL = ["momentum", "storage_change_anom", "hdd_anom"]      # deployable baseline (all lagged)
CORE_LAG = ["momentum", "storage_change_lag", "hdd_lag"]          # honest predetermined benchmark


def build_deployable(df: pd.DataFrame, horizon: int = HORIZON) -> pd.DataFrame:
    """DEPLOYABLE feature set: every candidate is its lagged anomaly (predetermined)."""
    out = df[["Date", "month", "TTF"]].copy()
    out["y"] = np.log(df["TTF"]).diff(horizon)
    for name in CANDIDATE_NAMES:
        m = CANDIDATE_META[name]
        series = TRANSFORMS[m["transform"]](df, m["col"])
        out[name] = series.shift(horizon)
    return out


def build_conditional(df: pd.DataFrame, horizon: int = HORIZON) -> pd.DataFrame:
    """
    CONDITIONAL feature set: momentum LAGGED (predetermined); every fundamental
    enters CONTEMPORANEOUSLY (perfect foresight of month t). Adds the lagged core
    terms used by the honest predetermined benchmark.
    """
    out = df[["Date", "month", "TTF"]].copy()
    out["y"] = np.log(df["TTF"]).diff(horizon)
    for name in CANDIDATE_NAMES:
        m = CANDIDATE_META[name]
        s = TRANSFORMS[m["transform"]](df, m["col"])
        shift = horizon if name == "momentum" else 0     # momentum lagged; fundamentals contemporaneous
        out[name] = s.shift(shift)
    out["storage_change_lag"] = change_anom(df, "EU+UK_av_storage(bcm)").shift(horizon)
    out["hdd_lag"] = level_anom(df, "Europe_HDD").shift(horizon)
    return out

In [24]:
# --------------------------------------------------------------------------- #
# EXOGENOUS-ceiling sub-model: pre-specified, endogeneity-free feature set
# --------------------------------------------------------------------------- #

# Truly-exogenous weather (price cannot cause these); storage is semi-endogenous.
WEATHER = ["hdd_anom", "cdd_anom", "wind_anom", "solar_anom", "precip_anom"]
EXO_WEATHER = ["momentum"] + WEATHER                    # + lagged momentum control
EXO_FULL = EXO_WEATHER + ["storage_change_anom"]        # add contemporaneous storage

EXO_SIGN = {"hdd_anom": +1, "cdd_anom": +1, "wind_anom": -1, "solar_anom": -1,
            "precip_anom": -1, "storage_change_anom": -1, "momentum": +1}


def build_exogenous(df, horizon=HORIZON):
    out = df[["Date", "month", "TTF"]].copy()
    out["y"] = np.log(df["TTF"]).diff(horizon)
    out["momentum"] = dlog_anom(df, "TTF").shift(horizon)          # predetermined
    # contemporaneous exogenous weather (perfect foresight)
    out["hdd_anom"] = level_anom(df, "Europe_HDD")
    out["cdd_anom"] = level_anom(df, "Europe_CDD")
    out["wind_anom"] = level_anom(df, "EU_wind_speed")
    out["solar_anom"] = level_anom(df, "EU_solar")
    out["precip_anom"] = level_anom(df, "Nordic_precip")
    # contemporaneous storage (semi-endogenous)
    out["storage_change_anom"] = change_anom(df, "EU+UK_av_storage(bcm)")
    # lagged core benchmark terms
    out["storage_change_lag"] = change_anom(df, "EU+UK_av_storage(bcm)").shift(horizon)
    out["hdd_lag"] = level_anom(df, "Europe_HDD").shift(horizon)
    return out


def regime_r2_exo(feat, cols):
    f = add_regime(feat)
    print("\nRegime R2 for the weather+storage ceiling model:")
    for rc, hi, lo in [("crisis", "crisis", "calm"), ("highvol", "high_vol", "low_vol")]:
        for val, lab in [(1, hi), (0, lo)]:
            d = f.loc[f[rc] == val, ["y"] + cols].dropna()
            if len(d) >= len(cols) + 5:
                rr = sm.OLS(d["y"], sm.add_constant(d[cols])).fit().rsquared
                print("  %-9s n=%3d  R2=%.3f" % (lab, len(d), rr))

In [25]:
# --------------------------------------------------------------------------- #
# LEAN sub-model: deliberately three-parameter, pre-specified
# --------------------------------------------------------------------------- #

LEAN = ["momentum", "storage_change_now", "hdd_now"]
LEAN_SIGN = {"momentum": +1, "storage_change_now": -1, "hdd_now": +1}


def build_lean(df, horizon=HORIZON):
    out = df[["Date", "month", "TTF"]].copy()
    out["y"] = np.log(df["TTF"]).diff(horizon)
    out["momentum"] = dlog_anom(df, "TTF").shift(horizon)                 # predetermined
    out["storage_change_now"] = change_anom(df, "EU+UK_av_storage(bcm)")  # contemporaneous
    out["hdd_now"] = level_anom(df, "Europe_HDD")                         # contemporaneous
    out["storage_change_lag"] = change_anom(df, "EU+UK_av_storage(bcm)").shift(horizon)
    out["hdd_lag"] = level_anom(df, "Europe_HDD").shift(horizon)
    return out

In [26]:
# --------------------------------------------------------------------------- #
# Sub-model runners
# --------------------------------------------------------------------------- #

def run_deployable():
    print("\n\n" + "#" * 78)
    print("# SUB-MODEL 1 of 4: DEPLOYABLE (predetermined -- every predictor lagged)")
    print("#" * 78)
    df = load_data()

    # Guardrail: make sure no global column ever sneaks into a candidate.
    used_cols = {CANDIDATE_META[n]["col"] for n in CANDIDATE_NAMES}
    leaked = used_cols.intersection(GLOBAL_COLS_EXCLUDED)
    assert not leaked, f"global column(s) leaked into candidates: {leaked}"

    feat = build_deployable(df, horizon=HORIZON)

    print("=" * 78)
    print("Stationarity gate (training sample, H=1 candidates)")
    print("=" * 78)
    rep, usable = stationarity_gate(feat, CANDIDATE_NAMES)
    print(rep.to_string(index=False))
    print(f"\nUsable (not clearly non-stationary): {len(usable)} of {len(CANDIDATE_NAMES)}")

    print("\n" + "=" * 78)
    print("Univariate HAC screen (training sample) -- ranked by |t|")
    print("=" * 78)
    screen = univariate_screen(feat, usable, CANDIDATE_META)
    show = screen[["name", "coef", "t", "p", "R2", "exp_sign", "sign_ok", "note"]]
    print(show.to_string(index=False))

    kept_pool, dropped = prune_collinear(feat, screen)
    if dropped:
        print("\nCollinearity pruning (|corr| > %.2f), dropped -> kept-instead:" % CORR_PRUNE_THRESH)
        for d, k in dropped.items():
            print(f"  {d:22} -> {k}")

    selected = select_model(screen, kept_pool)
    print("\nSelected compact European model:", selected)

    print("\n" + "=" * 78)
    print("In-sample HAC fit -- selected European model")
    print("=" * 78)
    eu_model = fit_hac(feat, selected)
    print(eu_model.summary())

    print("\n" + "=" * 78)
    print("Out-of-sample walk-forward (H=1): core vs European vs benchmarks")
    print("=" * 78)
    res = walk_forward(feat, {"core": CORE_MODEL, "european": selected})

    print("\n-- Full backtest window (%s to %s) --"
          % (res["Date"].min().date(), res["Date"].max().date()))
    print(summarize(res, ["core", "european"]).to_string(index=False))

    clean = res.loc[res["Date"] >= TRAIN_END]
    if len(clean) >= 6:
        print("\n-- Clean post-%s slice (selection never saw this) --" % TRAIN_END[:7])
        print(summarize(clean, ["core", "european"]).to_string(index=False))
        print("\n(DM t vs RW > ~1.65 one-sided ~ 10%%, > ~1.96 ~ 5%%. "
              "Small n post-%s, so treat as indicative.)" % TRAIN_END[:7])
    else:
        print("\n(Too few post-%s observations for a separate clean slice.)" % TRAIN_END[:7])

    dm_eu_vs_core = dm_tstat(res["actual"], res["pred_core"], res["pred_european"])
    print("\nDM t-stat, European vs core (positive => European lowers MSE): %.2f" % dm_eu_vs_core)

    regime_report_full(feat, selected)

In [27]:
def run_conditional_all():
    print("\n\n" + "#" * 78)
    print("# SUB-MODEL 2 of 4: CONDITIONAL -- perfect foresight, ALL fundamentals")
    print("#" * 78)
    df = load_data()
    feat = build_conditional(df, horizon=HORIZON)

    print("CONDITIONAL / PERFECT-FORESIGHT model -- fundamentals enter contemporaneously.")
    print("This is an EXPLANATORY CEILING, not a deployable forecast (see docstring).\n")

    print("=" * 78 + "\nStationarity gate\n" + "=" * 78)
    rep, usable = stationarity_gate(feat, CANDIDATE_NAMES)
    print(rep.to_string(index=False))
    print(f"\nUsable: {len(usable)} of {len(CANDIDATE_NAMES)}")

    print("\n" + "=" * 78 + "\nUnivariate HAC screen (contemporaneous) -- ranked by |t|\n" + "=" * 78)
    screen = univariate_screen(feat, usable, CANDIDATE_META)
    print(screen[["name", "coef", "t", "p", "R2", "exp_sign", "sign_ok"]].to_string(index=False))

    kept, dropped = prune_collinear(feat, screen)
    if dropped:
        print("\nCollinearity pruning (|corr| > %.2f):" % CORR_PRUNE_THRESH)
        for d, k in dropped.items():
            print(f"  {d:26} -> {k}")

    selected = select_model(screen, kept)
    print("\nSelected conditional model:", selected)

    print("\n" + "=" * 78 + "\nIn-sample HAC fit -- selected conditional model\n" + "=" * 78)
    m = fit_hac(feat, selected)
    print(m.summary())

    print("\n" + "=" * 78)
    print("Out-of-sample walk-forward (H=1): honest core (lagged) vs conditional (foresight)")
    print("=" * 78)
    res = walk_forward(feat, {"core_lag": CORE_LAG, "conditional": selected})
    print("\n-- Full backtest (%s to %s) --" % (res["Date"].min().date(), res["Date"].max().date()))
    print(summarize(res, ["core_lag", "conditional"]).to_string(index=False))

    clean = res.loc[res["Date"] >= TRAIN_END]
    if len(clean) >= 6:
        print("\n-- Clean post-%s slice --" % TRAIN_END[:7])
        print(summarize(clean, ["core_lag", "conditional"]).to_string(index=False))

    dm = dm_tstat(res["actual"], res["pred_core_lag"], res["pred_conditional"])
    print("\nDM t-stat, conditional vs honest core (positive => foresight lowers MSE): %.2f" % dm)
    print("\nNOTE: 'conditional' uses perfect foresight of same-month fundamentals -- it is an")
    print("upper bound on fundamental explanatory power, not an achievable live forecast.")

    regime_report_simple(feat, selected)

In [28]:
def run_exogenous_ceiling():
    print("\n\n" + "#" * 78)
    print("# SUB-MODEL 3 of 4: EXOGENOUS CEILING -- pre-specified weather + storage")
    print("#" * 78)
    df = load_data()
    feat = build_exogenous(df, horizon=HORIZON)

    print("ENDOGENEITY-FREE CEILING -- pre-specified exogenous model, perfect foresight.")
    print("Weather is truly exogenous; storage is semi-endogenous (flagged).\n")

    print("=" * 70)
    print("In-sample R2 decomposition (train %s..%s)" % (feat["Date"].min().date(), TRAIN_END[:7]))
    print("=" * 70)
    decomp = [
        ("momentum only (lagged)",            ["momentum"]),
        ("weather only (no momentum)",        WEATHER),
        ("weather + momentum",                EXO_WEATHER),
        ("weather + storage (no momentum)",   WEATHER + ["storage_change_anom"]),
        ("weather + storage + momentum",      EXO_FULL),
    ]
    for label, cols in decomp:
        rr, n = r2(feat, cols)
        print("  %-34s R2=%.3f   (n=%d)" % (label, rr, n))
    print("\n  (Compare: the earlier UNRESTRICTED perfect-foresight model hit R2~0.42, but")
    print("   leaned on wrong-signed endogenous vars. This is the clean causal ceiling.)")

    print("\n" + "=" * 70)
    print("HAC fit -- weather + storage + momentum (are the signs right?)")
    print("=" * 70)
    m = fit_hac(feat, EXO_FULL)
    print(m.summary())
    print("\nSign check (coef sign vs economic prior):")
    for c in EXO_FULL:
        ok = np.sign(m.params[c]) == EXO_SIGN[c]
        print("  %-22s coef=%+.5f  t=%+.2f  p=%.3f  sign_ok=%s"
              % (c, m.params[c], m.tvalues[c], m.pvalues[c], ok))

    print("\n" + "=" * 70)
    print("Out-of-sample walk-forward: honest core vs exogenous-foresight models")
    print("=" * 70)
    res = walk_forward(feat, {"core_lag": CORE_LAG, "exo_weather": EXO_WEATHER, "exo_full": EXO_FULL})
    print("\n-- Full backtest (%s to %s) --" % (res["Date"].min().date(), res["Date"].max().date()))
    print(summarize(res, ["core_lag", "exo_weather", "exo_full"]).to_string(index=False))
    clean = res.loc[res["Date"] >= TRAIN_END]
    if len(clean) >= 6:
        print("\n-- Clean post-%s slice --" % TRAIN_END[:7])
        print(summarize(clean, ["core_lag", "exo_weather", "exo_full"]).to_string(index=False))
    print("\nDM t, exo_full vs honest core (positive => foresight helps): %.2f"
          % dm_tstat(res["actual"], res["pred_core_lag"], res["pred_exo_full"]))

    regime_r2_exo(feat, EXO_FULL)
    print("\nNOTE: perfect-foresight ceiling; weather exogenous (clean), storage semi-endogenous.")

In [29]:
def run_lean():
    print("\n\n" + "#" * 78)
    print("# SUB-MODEL 4 of 4: LEAN -- 3-parameter perfect-foresight robustness check")
    print("#" * 78)
    df = load_data()
    feat = build_lean(df, horizon=HORIZON)

    print("LEAN perfect-foresight model (3 params): momentum(lag) + storage_now + HDD_now\n")

    print("=" * 66)
    print("In-sample R2 build-up (train %s..%s)" % (feat["Date"].min().date(), TRAIN_END[:7]))
    print("=" * 66)
    for label, cols in [("momentum only", ["momentum"]),
                        ("+ storage (now)", ["momentum", "storage_change_now"]),
                        ("+ HDD (now)  [= LEAN]", LEAN),
                        ("honest core (all lagged)", CORE_LAG)]:
        rr, n = r2(feat, cols)
        print("  %-28s R2=%.3f  (n=%d)" % (label, rr, n))

    print("\n" + "=" * 66)
    print("HAC fit -- LEAN model (signs & significance)")
    print("=" * 66)
    m = fit_hac(feat, LEAN)
    print(m.summary())
    for c in LEAN:
        print("  %-20s coef=%+.5f  t=%+.2f  p=%.3f  sign_ok=%s"
              % (c, m.params[c], m.tvalues[c], m.pvalues[c], np.sign(m.params[c]) == LEAN_SIGN[c]))

    print("\n" + "=" * 66)
    print("Out-of-sample walk-forward: LEAN vs honest core vs random walk")
    print("=" * 66)
    res = walk_forward(feat, {"core_lag": CORE_LAG, "lean": LEAN})
    print("\n-- Full backtest (%s to %s) --" % (res["Date"].min().date(), res["Date"].max().date()))
    print(summarize(res, ["core_lag", "lean"]).to_string(index=False))
    clean = res.loc[res["Date"] >= TRAIN_END]
    if len(clean) >= 6:
        print("\n-- Clean post-%s slice --" % TRAIN_END[:7])
        print(summarize(clean, ["core_lag", "lean"]).to_string(index=False))
    print("\nDM t, LEAN vs honest core (positive => foresight helps): %.2f"
          % dm_tstat(res["actual"], res["pred_core_lag"], res["pred_lean"]))

    # regime R2
    f = add_regime(feat)
    print("\nRegime R2 (LEAN model):")
    for rc, hi, lo in [("crisis", "crisis", "calm"), ("highvol", "high_vol", "low_vol")]:
        for val, lab in [(1, hi), (0, lo)]:
            d = f.loc[f[rc] == val, ["y"] + LEAN].dropna()
            if len(d) >= len(LEAN) + 5:
                rr = sm.OLS(d["y"], sm.add_constant(d[LEAN])).fit().rsquared
                print("  %-9s n=%3d  R2=%.3f" % (lab, len(d), rr))
    print("\nNOTE: perfect-foresight ceiling; storage semi-endogenous, HDD exogenous.")

In [30]:
# --------------------------------------------------------------------------- #
# Master entry point -- runs all four sub-models in sequence
# --------------------------------------------------------------------------- #

def main():
    run_deployable()
    run_conditional_all()
    run_exogenous_ceiling()
    run_lean()

In [31]:
# ---- run this section ----
main()



##############################################################################
# SUB-MODEL 1 of 4: DEPLOYABLE (predetermined -- every predictor lagged)
##############################################################################
Stationarity gate (training sample, H=1 candidates)
                     name        adf_p   kpss_p        verdict  usable
                 momentum 2.076585e-12 0.100000     stationary    True
      storage_change_anom 1.129799e-06 0.100000     stationary    True
       storage_level_anom 1.129812e-01 0.100000    conflicting    True
          production_anom 5.398473e-01 0.010000 non-stationary   False
           net_piped_anom 7.866081e-01 0.010000 non-stationary   False
         lng_imports_anom 1.456683e-01 0.010000 non-stationary   False
          net_supply_anom 7.789721e-01 0.010000 non-stationary   False
        total_demand_anom 1.750841e-01 0.010000 non-stationary   False
     nonpower_demand_anom 9.279556e-02 0.010000 non-stationary   False
    p

---

## Section 5 — Global gas modelling (Compact / conditional / lean / ECM / VECM)

Source: `Global_monthly_gas_model.py`

<details><summary>original module docstring</summary>

```
Global_monthly_gas_model.py

SINGLE consolidated script for the GLOBAL half of the month-ahead (H=1) TTF study
-- the companion to European_monthly_gas_model.py. It merges the separate global
scripts into one file and runs them in the order they feed the umbrella paper's
Section 5:

  §5.1  GLOBAL COMPACT (deployable)          [ttf_global_model.py]
  §5.2  GLOBAL CONDITIONAL ceiling           [ttf_global_conditional.py]
  §5.2  GLOBAL LEAN (NE-Asia DD isolation)   [ttf_global_lean.py]
  §5.3  ARDL/ECM deployable + cointegration  [ttf_ardl_ecm.py]
  §5.3  ECM ceiling / lean / regime          [ttf_ecm_ceiling.py]
  §5.4  VECM transmission -- HEADLINE run     [ttf_vecm.py, TTF-first ordering]
  §5.4  VECM transmission -- ROBUSTNESS run   [ttf_vecm.py, HH-first ordering]
  §5.4  VECM regime split (pre/post 2021-09) [ttf_vecm_split.py]

  All sub-models are now script-included and run for real -- no stubs remain.

Common design (shared by every global sub-model, exactly as in the European work):
  * Target y = one-month log return of front-month TTF (USD/mmbtu). H = 1.
  * Every predictor is a deseasonalized anomaly vs an EXPANDING, prior-years-only
    calendar-month climatology, so nothing uses future information.
  * momentum is always LAGGED (predetermined). In deployable mode all predictors
    are lagged; in conditional/ceiling mode the fundamentals enter contemporaneously
    (perfect foresight of QUANTITIES only -- never hub prices, which co-move with
    TTF and would be near-circular).
  * Selection/fit use data strictly before TRAIN_END; evaluation is an expanding
    walk-forward with a Newey-West Diebold-Mariano test vs a random walk, reported
    over the full backtest and the clean post-TRAIN_END slice.

Run:  python Global_monthly_gas_model.py   (runs every available global sub-model)

Requirements: pandas, numpy, statsmodels, scipy
```

</details>

In [32]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.vector_ar.vecm import (
    VECM, select_order, select_coint_rank, coint_johansen)
from statsmodels.tsa.api import VAR

In [33]:
# --------------------------------------------------------------------------- #
# Shared configuration
# --------------------------------------------------------------------------- #

DATA_PATH = "NG_m_final.csv"
TRAIN_END = "2025-01-01"
SAMPLE_START = "2015-01-01"     # JKM is real from 2015 (only a 4-month 2017 gap filled)
HORIZON = 1
MIN_TRAIN = 60                  # standard walk-forward warm-up (ceiling/lean/global studies)
ECM_MIN_TRAIN = 36             # ttf_ardl_ecm.py used a 36-month warm-up
CRISIS_START, CRISIS_END = "2021-09-01", "2023-07-01"
LONGRUN_X = ["log_JKM", "log_HH"]   # cointegrating partners for the ECM/ceiling

In [34]:
# --------------------------------------------------------------------------- #
# Shared helpers (byte-identical across the source scripts)
# --------------------------------------------------------------------------- #

def expanding_seasonal_mean(series, month):
    """Mean of all PRIOR years' values for the same calendar month (current excluded)."""
    out = pd.Series(index=series.index, dtype=float)
    tmp = pd.DataFrame({"val": series, "month": month})
    for _, grp in tmp.groupby("month"):
        out.loc[grp.index] = grp["val"].expanding().mean().shift(1)
    return out


def level_anom(df, col):
    return df[col] - expanding_seasonal_mean(df[col], df["month"])


def change_anom(df, col):
    ch = df[col].diff()
    return ch - expanding_seasonal_mean(ch, df["month"])


def dlog_anom(df, col):
    dl = np.log(df[col]).diff()
    return dl - expanding_seasonal_mean(dl, df["month"])


def hac_lags(n, extra=0):
    return int(np.floor(4 * (n / 100) ** (2 / 9))) + extra


def fit_hac(feat, cols, train_end=TRAIN_END):
    d = feat.loc[feat["Date"] < train_end, ["y"] + cols].dropna()
    return sm.OLS(d["y"], sm.add_constant(d[cols])).fit(
        cov_type="HAC", cov_kwds={"maxlags": hac_lags(len(d)), "use_correction": True})


def r2(feat, cols, train_end=TRAIN_END):
    d = feat.loc[feat["Date"] < train_end, ["y"] + cols].dropna()
    return sm.OLS(d["y"], sm.add_constant(d[cols])).fit().rsquared, len(d)


def dm_tstat(actual, pred_worse, pred_better, h=HORIZON):
    """Diebold-Mariano statistic (squared-error loss, Newey-West variance, h-1 lags)."""
    a = np.asarray(actual, float)
    d = (a - np.asarray(pred_worse, float)) ** 2 - (a - np.asarray(pred_better, float)) ** 2
    d = d[~np.isnan(d)]
    n = len(d); dbar = d.mean(); lag = max(h - 1, 0)
    var = ((d - dbar) ** 2).mean()
    for l in range(1, lag + 1):
        var += 2 * (1 - l / (lag + 1)) * ((d[l:] - dbar) * (d[:-l] - dbar)).mean()
    return dbar / np.sqrt(var / n) if var > 0 else np.nan


def summarize(res, model_names):
    rw = res["pred_random_walk"]; rw_err = res["actual"] - rw
    rows = []
    for name in model_names + ["random_walk", "hist_mean"]:
        pred = res[f"pred_{name}"]; err = res["actual"] - pred
        rows.append({"Model": name, "RMSE": np.sqrt((err ** 2).mean()),
                     "Hit rate": np.nan if name == "random_walk" else (np.sign(res["actual"]) == np.sign(pred)).mean(),
                     "OOS R2 vs RW": np.nan if name == "random_walk" else 1 - (err ** 2).sum() / (rw_err ** 2).sum(),
                     "DM t vs RW": np.nan if name == "random_walk" else dm_tstat(res["actual"], rw, pred),
                     "n": len(res)})
    return pd.DataFrame(rows)


def walk_forward_fixed(feat, model_cols, min_train=MIN_TRAIN):
    """Fixed-feature-set expanding walk-forward (global conditional & lean models)."""
    all_cols = sorted({c for cols in model_cols.values() for c in cols})
    data = feat[["Date", "y"] + all_cols].dropna().reset_index(drop=True)
    rows = []
    for i in range(min_train, len(data)):
        train, test = data.iloc[:i], data.iloc[i]
        rec = {"Date": test["Date"], "actual": test["y"],
               "pred_random_walk": 0.0, "pred_hist_mean": train["y"].mean()}
        for name, cols in model_cols.items():
            fit = sm.OLS(train["y"], sm.add_constant(train[cols])).fit()
            Xte = sm.add_constant(test[cols].to_frame().T, has_constant="add")
            rec[f"pred_{name}"] = fit.predict(Xte).iloc[0]
        rows.append(rec)
    return pd.DataFrame(rows)


def regime_r2(feat, cols, label):
    """Crisis/calm and high/low-vol in-sample R2 (global ceiling, lean, ECM ceiling/lean)."""
    f = feat.copy()
    f["crisis"] = ((f["Date"] >= CRISIS_START) & (f["Date"] < CRISIS_END)).astype(int)
    vol = f["y"].rolling(6).std().shift(1)
    f["highvol"] = (vol > vol.expanding().median().shift(1)).astype(float)
    print("\nRegime R2 (%s):" % label)
    for rc, hi, lo in [("crisis", "crisis", "calm"), ("highvol", "high_vol", "low_vol")]:
        for val, lab in [(1, hi), (0, lo)]:
            d = f.loc[f[rc] == val, ["y"] + cols].dropna()
            if len(d) >= len(cols) + 5:
                rr = sm.OLS(d["y"], sm.add_constant(d[cols])).fit().rsquared
                print("  %-9s n=%3d  R2=%.3f" % (lab, len(d), rr))


def long_run_ols(sub):
    """Engle-Granger step 1: OLS of log_TTF on the cointegrating partners."""
    return sm.OLS(sub["log_TTF"], sm.add_constant(sub[LONGRUN_X])).fit()


def ect_from_fit(frame, lr_fit):
    """Equilibrium error u_t = log_TTF_t - fitted long-run level."""
    X = sm.add_constant(frame[LONGRUN_X], has_constant="add")
    return frame["log_TTF"] - lr_fit.predict(X)

In [35]:
# --------------------------------------------------------------------------- #
# Data loaders
# --------------------------------------------------------------------------- #

def load_data_simple(path=DATA_PATH):
    """For the naive-OLS global ceiling/lean models (no hub-price logs needed)."""
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df["month"] = df["Date"].dt.month
    df["TTF"] = df["TTF(USD/mmbtu)"]
    return df


def load_data_prices(path=DATA_PATH):
    """For the ECM models: adds log hub prices; interpolates the 4-month 2017 JKM gap."""
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df["month"] = df["Date"].dt.month
    df["TTF"] = df["TTF(USD/mmbtu)"]
    jkm = df["JKM(USD/mmbtu)"].astype(float).copy()
    jkm[np.isclose(jkm, 7.86)] = np.nan
    df["JKM(USD/mmbtu)"] = np.exp(np.log(jkm).interpolate(limit_direction="both"))
    df["log_TTF"] = np.log(df["TTF(USD/mmbtu)"])
    df["log_JKM"] = np.log(df["JKM(USD/mmbtu)"])
    df["log_HH"] = np.log(df["HH(USD/mmbtu)"])
    return df


# =========================================================================== #
# §5.1  GLOBAL COMPACT (deployable)  -- ttf_global_model.py
# =========================================================================== #
# Widened ~55-predictor pool (EU + Global blocks + momentum), ALL lagged, run
# through the stationarity gate -> univariate HAC screen -> collinearity prune ->
# training-only selection, benchmarked vs the European core, with a joint-Wald
# regime split. Reuses the merge's shared level/change/dlog anomaly builders,
# hac_lags, dm_tstat, summarize and walk_forward_fixed; compact-specific
# machinery is prefixed gm_ / GM_.

In [36]:
GM_N_SELECT = 6
GM_CORR_PRUNE_THRESH = 0.70
GM_SCREEN_P_MAX = 0.10
GM_TRANSFORMS = {"level_anom": level_anom, "change_anom": change_anom, "dlog_anom": dlog_anom}

# (name, source column, transform, expected sign, note, block in {Own, EU, Global})
GM_CANDIDATES = [
    ("momentum",              "TTF",                       "dlog_anom",  +1, "own 1-month log return", "Own"),
    # ============================ EUROPEAN BLOCK ============================
    ("storage_change_anom",   "EU+UK_av_storage(bcm)",     "change_anom", -1, "MoM storage change (build = bearish)", "EU"),
    ("storage_level_anom",    "EU+UK_av_storage(bcm)",     "level_anom",  -1, "storage level", "EU"),
    ("production_anom",       "EU+UK Production(bcm)",      "level_anom",  -1, "indigenous production", "EU"),
    ("net_piped_anom",        "EU+UK Net_piped(bcm)",      "level_anom",  -1, "net pipeline imports", "EU"),
    ("lng_imports_anom",      "EU+UK LNG imports",         "level_anom",  -1, "EU+UK LNG imports (supply)", "EU"),
    ("net_supply_anom",       "EU+UK Net_supply",          "level_anom",  -1, "total net supply", "EU"),
    ("total_demand_anom",     "EU+UK Total(bcm)",          "level_anom",  +1, "total gas demand", "EU"),
    ("nonpower_demand_anom",  "EU+UK Non_power(bcm)",       "level_anom",  +1, "non-power gas demand", "EU"),
    ("power_gas_demand_anom", "EU+UK Electricity(bcm)",    "level_anom",  +1, "gas-for-power demand (bcm)", "EU"),
    ("residual_load_anom",    "EU+UK Residual load",       "level_anom",  +1, "thermal power demand", "EU"),
    ("coal_gen_anom",         "EU+UK Coal",                "level_anom",  -1, "coal generation", "EU"),
    ("nuclear_gen_anom",      "EU+UK Nuclear",             "level_anom",  -1, "nuclear generation", "EU"),
    ("hydro_gen_anom",        "EU+UK Hydro_gen",           "level_anom",  -1, "hydro generation", "EU"),
    ("gas_burn_anom",         "EU+UK Fossil gas",          "level_anom",  +1, "gas-fired generation (endogenous)", "EU"),
    ("production_chg_anom",     "EU+UK Production(bcm)",    "change_anom", -1, "change in indigenous production", "EU"),
    ("net_piped_chg_anom",      "EU+UK Net_piped(bcm)",    "change_anom", -1, "change in net pipeline imports", "EU"),
    ("lng_imports_chg_anom",    "EU+UK LNG imports",       "change_anom", -1, "change in EU+UK LNG imports", "EU"),
    ("net_supply_chg_anom",     "EU+UK Net_supply",        "change_anom", -1, "change in total net supply", "EU"),
    ("total_demand_chg_anom",   "EU+UK Total(bcm)",        "change_anom", +1, "change in total gas demand", "EU"),
    ("nonpower_demand_chg_anom","EU+UK Non_power(bcm)",     "change_anom", +1, "change in non-power gas demand", "EU"),
    ("power_gas_demand_chg_anom","EU+UK Electricity(bcm)",  "change_anom", +1, "change in gas-for-power demand", "EU"),
    ("residual_load_chg_anom",  "EU+UK Residual load",     "change_anom", +1, "change in thermal power demand", "EU"),
    ("coal_gen_chg_anom",       "EU+UK Coal",              "change_anom", -1, "change in coal generation", "EU"),
    ("nuclear_gen_chg_anom",    "EU+UK Nuclear",           "change_anom", -1, "change in nuclear generation", "EU"),
    ("hydro_gen_chg_anom",      "EU+UK Hydro_gen",         "change_anom", -1, "change in hydro generation", "EU"),
    ("gas_burn_chg_anom",       "EU+UK Fossil gas",        "change_anom", +1, "change in gas-fired generation (endogenous)", "EU"),
    ("norway_prod_anom",      "Norway_gas_prod",           "level_anom",  -1, "Norwegian gas production", "EU"),
    ("norway_supplyred_anom", "Norway_supply_red",         "level_anom",  +1, "Norway supply reduction", "EU"),
    ("norway_planned_anom",   "Norway_planned_outage",     "level_anom",  +1, "Norway planned outages", "EU"),
    ("norway_unplanned_anom", "Norway_unplanned_outage",   "level_anom",  +1, "Norway unplanned outages", "EU"),
    ("hdd_anom",              "Europe_HDD",                "level_anom",  +1, "European heating degree days", "EU"),
    ("cdd_anom",              "Europe_CDD",                "level_anom",  +1, "European cooling degree days", "EU"),
    ("wind_anom",             "EU_wind_speed",             "level_anom",  -1, "wind speed (more wind = less gas)", "EU"),
    ("solar_anom",            "EU_solar",                  "level_anom",  -1, "solar irradiation", "EU"),
    ("precip_anom",           "Nordic_precip",             "level_anom",  -1, "Nordic precipitation (hydro inflows)", "EU"),
    # ============================ GLOBAL BLOCK =============================
    ("jkm_mom",               "JKM(USD/mmbtu)",            "dlog_anom",   +1, "JKM (Asia) 1-month return", "Global"),
    ("hh_mom",                "HH(USD/mmbtu)",             "dlog_anom",   +1, "Henry Hub (US) 1-month return", "Global"),
    ("ttf_jkm_spread_anom",   "ttf_jkm_logspread",         "level_anom",  -1, "TTF-JKM log spread (arbitrage; mean-reverting)", "Global"),
    ("ttf_hh_spread_anom",    "ttf_hh_logspread",          "level_anom",  -1, "TTF-HH log spread (US export arbitrage)", "Global"),
    ("ttf_jkm_spread_chg_anom","ttf_jkm_logspread",        "change_anom", -1, "change in TTF-JKM spread", "Global"),
    ("us_gwdd_anom",          "US_GWDD",                   "level_anom",  +1, "US degree days (US demand -> less export)", "Global"),
    ("neasia_gwdd_anom",      "NE_Asia_GWDD",              "level_anom",  +1, "NE Asia degree days (LNG demand pull)", "Global"),
    ("atlantic_ace_anom",     "Atlantic_ACE",              "level_anom",  +1, "Atlantic hurricane energy (Gulf LNG risk)", "Global"),
    ("gulf_storm_anom",       "Gulf_storm_days",           "level_anom",  +1, "Gulf storm days (US LNG export disruption)", "Global"),
    ("global_lng_offline_anom",     "Global LNG capacity offline", "level_anom",  +1, "global LNG capacity offline (supply loss)", "Global"),
    ("global_lng_offline_chg_anom", "Global LNG capacity offline", "change_anom", +1, "change in global LNG offline", "Global"),
    ("global_lng_capacity_chg_anom","Global LNG nameplate capacity","change_anom", -1, "change in global LNG nameplate capacity", "Global"),
    ("asia_imports_chg_anom", "CH+JP+KR LNG imports",      "change_anom", +1, "change in China/Japan/Korea LNG imports (demand)", "Global"),
    ("india_imports_chg_anom","IN LNG imports",            "change_anom", +1, "change in India LNG imports", "Global"),
    ("qaus_exports_chg_anom", "QA+AU+US LNG exports",      "change_anom", -1, "change in Qatar/Australia/US LNG exports (supply)", "Global"),
    ("seasia_exports_chg_anom","ID+MY+BN LNG exports",     "change_anom", -1, "change in Indonesia/Malaysia/Brunei LNG exports", "Global"),
    ("nigeria_exports_chg_anom","NG LNG exports",          "change_anom", -1, "change in Nigeria LNG exports", "Global"),
    ("vix_anom",              "VIX",                       "level_anom",  +1, "equity volatility (risk sentiment)", "Global"),
    ("fx_ret_anom",           "USD-EUR_FX",                "dlog_anom",   +1, "USD-EUR FX return", "Global"),
]

In [37]:
GM_CANDIDATE_NAMES = [c[0] for c in GM_CANDIDATES]
GM_CANDIDATE_META = {c[0]: {"col": c[1], "transform": c[2], "sign": c[3], "note": c[4], "block": c[5]}
                     for c in GM_CANDIDATES}
GM_CORE_MODEL = ["momentum", "storage_change_anom", "hdd_anom"]   # established European baseline


def gm_load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df["month"] = df["Date"].dt.month
    df["TTF"] = df["TTF(USD/mmbtu)"]
    # LNG-arbitrage log spreads (JKM constant-backfilled pre-2019 -> spread unreliable then)
    df["ttf_jkm_logspread"] = np.log(df["TTF(USD/mmbtu)"]) - np.log(df["JKM(USD/mmbtu)"])
    df["ttf_hh_logspread"] = np.log(df["TTF(USD/mmbtu)"]) - np.log(df["HH(USD/mmbtu)"])
    return df


def gm_build_candidates(df, horizon=HORIZON):
    out = df[["Date", "month", "TTF"]].copy()
    out["y"] = np.log(df["TTF"]).diff(horizon)
    for name in GM_CANDIDATE_NAMES:
        m = GM_CANDIDATE_META[name]
        out[name] = GM_TRANSFORMS[m["transform"]](df, m["col"]).shift(horizon)
    return out

In [38]:
def gm_check_stationarity(series, name=""):
    s = series.dropna()
    adf_p = adfuller(s, autolag="AIC")[1]
    kpss_p = kpss(s, regression="c", nlags="auto")[1]
    stationary = (adf_p < 0.05) and (kpss_p > 0.05)
    borderline = (adf_p < 0.05) or (kpss_p > 0.05)
    verdict = "stationary" if stationary else ("conflicting" if borderline else "non-stationary")
    return {"name": name, "adf_p": adf_p, "kpss_p": kpss_p, "verdict": verdict, "usable": borderline}


def gm_stationarity_gate(feat, names, train_end=TRAIN_END):
    train = feat.loc[feat["Date"] < train_end]
    rep = pd.DataFrame([gm_check_stationarity(train[nm], nm) for nm in names])
    return rep, rep.loc[rep["usable"], "name"].tolist()


def gm_univariate_screen(feat, names, train_end=TRAIN_END):
    train = feat.loc[feat["Date"] < train_end]
    rows = []
    for nm in names:
        d = train[["y", nm]].dropna()
        if len(d) < 30:
            continue
        X = sm.add_constant(d[[nm]])
        m = sm.OLS(d["y"], X).fit(cov_type="HAC",
                                  cov_kwds={"maxlags": hac_lags(len(d)), "use_correction": True})
        rows.append({"name": nm, "block": GM_CANDIDATE_META[nm]["block"], "coef": m.params[nm],
                     "t": m.tvalues[nm], "p": m.pvalues[nm], "R2": m.rsquared,
                     "exp_sign": GM_CANDIDATE_META[nm]["sign"],
                     "sign_ok": np.sign(m.params[nm]) == GM_CANDIDATE_META[nm]["sign"],
                     "note": GM_CANDIDATE_META[nm]["note"]})
    out = pd.DataFrame(rows)
    return out.reindex(out["t"].abs().sort_values(ascending=False).index).reset_index(drop=True)

In [39]:
def gm_prune_collinear(feat, screen, train_end=TRAIN_END, thresh=GM_CORR_PRUNE_THRESH):
    train = feat.loc[feat["Date"] < train_end]
    ranked = screen["name"].tolist()
    corr = train[ranked].corr().abs()
    kept, dropped = [], {}
    for nm in ranked:
        clash = next((k for k in kept if corr.loc[nm, k] > thresh), None)
        if clash is None:
            kept.append(nm)
        else:
            dropped[nm] = clash
    return kept, dropped


def gm_select_model(screen, kept_pool, n_select=GM_N_SELECT, p_max=GM_SCREEN_P_MAX):
    ordered = [nm for nm in screen["name"].tolist() if nm in kept_pool]
    sig = set(screen.loc[screen["p"] < p_max, "name"])
    chosen = ["momentum"] if "momentum" in kept_pool else []
    for nm in ordered:
        if len(chosen) >= n_select:
            break
        if nm == "momentum" or nm not in sig:
            continue
        chosen.append(nm)
    return chosen

In [40]:
def gm_fit_hac(feat, cols, train_end=TRAIN_END):
    train = feat.loc[feat["Date"] < train_end, ["Date", "y"] + cols].dropna()
    X = sm.add_constant(train[cols])
    m = sm.OLS(train["y"], X).fit(cov_type="HAC",
                                  cov_kwds={"maxlags": hac_lags(len(train)), "use_correction": True})
    return m, train


def gm_add_regime(feat):
    f = feat.copy()
    f["crisis"] = ((f["Date"] >= CRISIS_START) & (f["Date"] < CRISIS_END)).astype(int)
    vol = f["y"].rolling(6).std().shift(1)
    med = vol.expanding().median().shift(1)
    f["highvol"] = (vol > med).astype(float)
    return f


def gm_subsample_fit(feat, cols, mask):
    d = feat.loc[mask, ["y"] + cols].dropna()
    if len(d) < len(cols) + 5:
        return None, d
    X = sm.add_constant(d[cols])
    return sm.OLS(d["y"], X).fit(cov_type="HAC",
                                 cov_kwds={"maxlags": hac_lags(len(d)), "use_correction": True}), d

In [41]:
def gm_regime_subsample(feat, cols, regime_col, hi_label, lo_label):
    tab = {c: {} for c in ["const"] + cols}
    meta = {}
    for val, lab in [(1, hi_label), (0, lo_label)]:
        m, d = gm_subsample_fit(feat, cols, feat[regime_col] == val)
        meta[lab] = {"n": len(d), "R2": (m.rsquared if m is not None else np.nan)}
        for c in ["const"] + cols:
            tab[c][f"coef[{lab}]"] = (m.params[c] if m is not None else np.nan)
            tab[c][f"t[{lab}]"] = (m.tvalues[c] if m is not None else np.nan)
    dfo = pd.DataFrame(tab).T[[f"coef[{hi_label}]", f"t[{hi_label}]", f"coef[{lo_label}]", f"t[{lo_label}]"]]
    return dfo.round(4), meta


def gm_regime_interaction(feat, cols, regime_col):
    d = feat[["y", regime_col] + cols].dropna().copy()
    X = pd.DataFrame({"regime": d[regime_col].astype(float)}, index=d.index)
    inter = []
    for c in cols:
        X[c] = d[c].values
        ic = f"{c}_Xreg"
        X[ic] = (d[c] * d[regime_col]).values
        inter.append(ic)
    X = sm.add_constant(X)
    m = sm.OLS(d["y"], X).fit(cov_type="HAC",
                              cov_kwds={"maxlags": hac_lags(len(d)), "use_correction": True})
    names = list(X.columns)
    R = np.zeros((len(inter), len(names)))
    for i, ic in enumerate(inter):
        R[i, names.index(ic)] = 1.0
    w = m.wald_test(R, use_f=True)
    inter_t = {c: m.tvalues[f"{c}_Xreg"] for c in cols}
    return m, inter_t, float(np.squeeze(w.statistic)), float(np.squeeze(w.pvalue)), len(d)

In [42]:
def gm_regime_report(feat, cols):
    feat = gm_add_regime(feat)
    print("\n" + "=" * 78)
    print("Regime split -- selected global model: %s" % cols)
    print("=" * 78)
    for regime_col, hi, lo in [("crisis", "crisis", "calm"), ("highvol", "high_vol", "low_vol")]:
        tag = ("Calendar crisis %s..%s" % (CRISIS_START[:7], CRISIS_END[:7])
               if regime_col == "crisis" else "Trailing-volatility (look-ahead-safe)")
        print("\n--- %s ---" % tag)
        sub, meta = gm_regime_subsample(feat, cols, regime_col, hi, lo)
        for lab, mm in meta.items():
            print(f"  {lab:9}: n={mm['n']:3d}   R2={mm['R2']:.3f}")
        print(sub.to_string())
        m, inter_t, wstat, wp, npool = gm_regime_interaction(feat, cols, regime_col)
        print("  interaction t-stats (coef differs in %s vs %s):" % (hi, lo))
        for c, t in inter_t.items():
            print(f"    {c:26} t = {t:+.2f}")
        print("  Joint Wald (all interactions = 0):  F = %.2f,  p = %.3f   [n=%d]" % (wstat, wp, npool))
        print("  => %s" % ("coefficients ARE regime-dependent" if wp < 0.10
                           else "no significant regime dependence detected"))

In [43]:
def run_global_compact():
    print("\n\n" + "#" * 78)
    print("# §5.1  GLOBAL COMPACT (deployable, all-lagged pool)  [ttf_global_model.py]")
    print("#" * 78)
    df = gm_load_data()
    feat = gm_build_candidates(df, horizon=HORIZON)
    n_eu = sum(1 for n in GM_CANDIDATE_NAMES if GM_CANDIDATE_META[n]["block"] == "EU")
    n_gl = sum(1 for n in GM_CANDIDATE_NAMES if GM_CANDIDATE_META[n]["block"] == "Global")
    print("Candidate pool: %d total  (%d European, %d Global, 1 Own/momentum)"
          % (len(GM_CANDIDATE_NAMES), n_eu, n_gl))

    print("\n" + "=" * 78)
    print("Stationarity gate (training sample, H=1 candidates)")
    print("=" * 78)
    rep, usable = gm_stationarity_gate(feat, GM_CANDIDATE_NAMES)
    print(rep.to_string(index=False))
    print(f"\nUsable (not clearly non-stationary): {len(usable)} of {len(GM_CANDIDATE_NAMES)}")

    print("\n" + "=" * 78)
    print("Univariate HAC screen (training sample) -- ranked by |t|")
    print("=" * 78)
    screen = gm_univariate_screen(feat, usable)
    print(screen[["name", "block", "coef", "t", "p", "R2", "exp_sign", "sign_ok"]].to_string(index=False))

    kept_pool, dropped = gm_prune_collinear(feat, screen)
    if dropped:
        print("\nCollinearity pruning (|corr| > %.2f), dropped -> kept-instead:" % GM_CORR_PRUNE_THRESH)
        for d, k in dropped.items():
            print(f"  {d:26} -> {k}")

    selected = gm_select_model(screen, kept_pool)
    blocks = [GM_CANDIDATE_META[c]["block"] for c in selected]
    print("\nSelected compact GLOBAL model:", selected)
    print("  block composition:  %d Own, %d EU, %d Global"
          % (blocks.count("Own"), blocks.count("EU"), blocks.count("Global")))

    print("\n" + "=" * 78)
    print("In-sample HAC fit -- selected global model")
    print("=" * 78)
    gl_model, _ = gm_fit_hac(feat, selected)
    print(gl_model.summary())

    print("\n" + "=" * 78)
    print("Out-of-sample walk-forward (H=1): European core vs Global vs benchmarks")
    print("=" * 78)
    res = walk_forward_fixed(feat, {"core": GM_CORE_MODEL, "global": selected})
    print("\n-- Full backtest window (%s to %s) --"
          % (res["Date"].min().date(), res["Date"].max().date()))
    print(summarize(res, ["core", "global"]).to_string(index=False))
    clean = res.loc[res["Date"] >= TRAIN_END]
    if len(clean) >= 6:
        print("\n-- Clean post-%s slice (selection never saw this) --" % TRAIN_END[:7])
        print(summarize(clean, ["core", "global"]).to_string(index=False))
        print("\n(DM t vs RW > ~1.65 one-sided ~ 10%%, > ~1.96 ~ 5%%. Small n post-%s.)" % TRAIN_END[:7])
    dm_gl_vs_core = dm_tstat(res["actual"], res["pred_core"], res["pred_global"])
    print("\nDM t-stat, Global vs European core (positive => Global lowers MSE): %.2f" % dm_gl_vs_core)

    gm_regime_report(feat, selected)


# =========================================================================== #
# §5.2  GLOBAL CONDITIONAL ceiling  -- ttf_global_conditional.py
# =========================================================================== #

# European clean ceiling (perfect foresight): storage + truly-exogenous weather.

In [44]:
EU_CLEAN = ["storage_now", "hdd_now", "cdd_now", "wind_now", "solar_now", "precip_now"]
# Global block price genuinely CANNOT cause -> clean causal foresight.
GLOBAL_EXO = ["glng_offline_now", "us_gwdd_now", "neasia_gwdd_now",
              "atlantic_ace_now", "gulf_storm_now"]
# Global block that RESPONDS to price (endogenous) -> reported separately.
GLOBAL_TRADE = ["asia_imp_chg_now", "india_imp_chg_now", "qaus_exp_chg_now",
                "seasia_exp_chg_now", "nigeria_exp_chg_now", "glng_capacity_chg_now"]
CONTROL = ["momentum"]
GC_CORE_LAG = ["momentum", "storage_lag", "hdd_lag"]
EU_CEIL = CONTROL + EU_CLEAN
GLOBAL_CLEAN = CONTROL + EU_CLEAN + GLOBAL_EXO
GLOBAL_FULL = GLOBAL_CLEAN + GLOBAL_TRADE

GC_SIGN = {
    "momentum": +1,
    "storage_now": -1, "hdd_now": +1, "cdd_now": +1,
    "wind_now": -1, "solar_now": -1, "precip_now": -1,
    "glng_offline_now": +1, "us_gwdd_now": +1, "neasia_gwdd_now": +1,
    "atlantic_ace_now": +1, "gulf_storm_now": +1,
    "asia_imp_chg_now": +1, "india_imp_chg_now": +1,
    "qaus_exp_chg_now": -1, "seasia_exp_chg_now": -1, "nigeria_exp_chg_now": -1,
    "glng_capacity_chg_now": -1,
}

In [45]:
def gc_build_features(df, horizon=HORIZON):
    out = df[["Date", "month", "TTF"]].copy()
    out["y"] = np.log(df["TTF"]).diff(horizon)
    out["momentum"] = dlog_anom(df, "TTF").shift(horizon)
    # European clean block, contemporaneous (perfect foresight)
    out["storage_now"] = change_anom(df, "EU+UK_av_storage(bcm)")
    out["hdd_now"] = level_anom(df, "Europe_HDD")
    out["cdd_now"] = level_anom(df, "Europe_CDD")
    out["wind_now"] = level_anom(df, "EU_wind_speed")
    out["solar_now"] = level_anom(df, "EU_solar")
    out["precip_now"] = level_anom(df, "Nordic_precip")
    # Global exogenous block, contemporaneous
    out["glng_offline_now"] = level_anom(df, "Global LNG capacity offline")
    out["us_gwdd_now"] = level_anom(df, "US_GWDD")
    out["neasia_gwdd_now"] = level_anom(df, "NE_Asia_GWDD")
    out["atlantic_ace_now"] = level_anom(df, "Atlantic_ACE")
    out["gulf_storm_now"] = level_anom(df, "Gulf_storm_days")
    # Global trade block, contemporaneous (endogenous)
    out["asia_imp_chg_now"] = change_anom(df, "CH+JP+KR LNG imports")
    out["india_imp_chg_now"] = change_anom(df, "IN LNG imports")
    out["qaus_exp_chg_now"] = change_anom(df, "QA+AU+US LNG exports")
    out["seasia_exp_chg_now"] = change_anom(df, "ID+MY+BN LNG exports")
    out["nigeria_exp_chg_now"] = change_anom(df, "NG LNG exports")
    out["glng_capacity_chg_now"] = change_anom(df, "Global LNG nameplate capacity")
    # lagged core benchmark terms
    out["storage_lag"] = change_anom(df, "EU+UK_av_storage(bcm)").shift(horizon)
    out["hdd_lag"] = level_anom(df, "Europe_HDD").shift(horizon)
    return out

In [46]:
def run_global_conditional():
    print("\n\n" + "#" * 78)
    print("# §5.2  GLOBAL CONDITIONAL ceiling (pre-specified, exogenous vs endogenous)")
    print("#" * 78)
    df = load_data_simple()
    feat = gc_build_features(df, horizon=HORIZON)

    print("GLOBAL perfect-foresight ceiling -- pre-specified, exogenous vs endogenous.")
    print("Foresight is granted to QUANTITIES only (storage, weather, LNG outages/trade);")
    print("hub prices (JKM/HH) are excluded -- knowing them contemporaneously is near-circular.")
    print("momentum is the only lagged control. H=1. European core is the honest benchmark.\n")

    print("=" * 74)
    print("In-sample R2 build-up (train %s..%s)" % (feat["Date"].min().date(), TRAIN_END[:7]))
    print("=" * 74)
    decomp = [
        ("momentum only (lagged)",                 CONTROL),
        ("+ EU clean ceiling (foresight)",         EU_CEIL),
        ("+ GLOBAL_EXO  [= global clean ceiling]", GLOBAL_CLEAN),
        ("+ GLOBAL_TRADE (endogenous)",            GLOBAL_FULL),
        ("(ref) honest EU core (all lagged)",      GC_CORE_LAG),
    ]
    for label, cols in decomp:
        rr, n = r2(feat, cols)
        print("  %-40s R2=%.3f  (n=%d)" % (label, rr, n))
    print("\n  Read-off: EU-clean is the established European ceiling (~0.21). What GLOBAL_EXO")
    print("  adds is the clean marginal contribution of global fundamentals; GLOBAL_TRADE's")
    print("  jump (if any) is mostly endogeneity (price pulling cargoes), not forecast power.")

    print("\n" + "=" * 74)
    print("HAC fit -- GLOBAL CLEAN ceiling (momentum + EU clean + GLOBAL_EXO): signs?")
    print("=" * 74)
    m = fit_hac(feat, GLOBAL_CLEAN)
    print(m.summary())
    print("\nSign check (coef sign vs economic prior):")
    for c in GLOBAL_CLEAN:
        ok = np.sign(m.params[c]) == GC_SIGN[c]
        print("  %-22s coef=%+.5f  t=%+.2f  p=%.3f  sign_ok=%s"
              % (c, m.params[c], m.tvalues[c], m.pvalues[c], ok))

    print("\n" + "=" * 74)
    print("HAC fit -- GLOBAL_EXO terms only, net of EU clean (isolated global signal)")
    print("=" * 74)
    mx = fit_hac(feat, GLOBAL_CLEAN)
    for c in GLOBAL_EXO:
        print("  %-22s coef=%+.5f  t=%+.2f  p=%.3f  sign_ok=%s"
              % (c, mx.params[c], mx.tvalues[c], mx.pvalues[c], np.sign(mx.params[c]) == GC_SIGN[c]))

    print("\n" + "=" * 74)
    print("Out-of-sample walk-forward: EU core vs EU ceiling vs global ceiling/full")
    print("=" * 74)
    res = walk_forward_fixed(feat, {"core_lag": GC_CORE_LAG, "eu_ceiling": EU_CEIL,
                                    "global_clean": GLOBAL_CLEAN, "global_full": GLOBAL_FULL})
    print("\n-- Full backtest (%s to %s) --" % (res["Date"].min().date(), res["Date"].max().date()))
    print(summarize(res, ["core_lag", "eu_ceiling", "global_clean", "global_full"]).to_string(index=False))
    clean = res.loc[res["Date"] >= TRAIN_END]
    if len(clean) >= 6:
        print("\n-- Clean post-%s slice (never selected on) --" % TRAIN_END[:7])
        print(summarize(clean, ["core_lag", "eu_ceiling", "global_clean", "global_full"]).to_string(index=False))

    print("\nDM t, global_clean vs EU ceiling (positive => global fundamentals help): %.2f"
          % dm_tstat(res["actual"], res["pred_eu_ceiling"], res["pred_global_clean"]))
    print("DM t, global_clean vs honest EU core (positive => global helps): %.2f"
          % dm_tstat(res["actual"], res["pred_core_lag"], res["pred_global_clean"]))

    regime_r2(feat, GLOBAL_CLEAN, "global clean ceiling")
    print("\nNOTE: perfect-foresight ceiling; GLOBAL_EXO exogenous (clean), storage semi-endogenous,")
    print("GLOBAL_TRADE endogenous (price-responsive). Not a deployable forecast.")


# =========================================================================== #
# §5.2  GLOBAL LEAN (NE-Asia DD isolation)  -- ttf_global_lean.py
# =========================================================================== #

In [47]:
GL_EU_LEAN = ["momentum", "storage_now", "hdd_now"]
GL_GLOBAL_LEAN = GL_EU_LEAN + ["neasia_gwdd_now"]
GL_CORE_LAG = ["momentum", "storage_lag", "hdd_lag"]
GL_SIGN = {"momentum": +1, "storage_now": -1, "hdd_now": +1, "neasia_gwdd_now": +1}


def gl_build_features(df, horizon=HORIZON):
    out = df[["Date", "month", "TTF"]].copy()
    out["y"] = np.log(df["TTF"]).diff(horizon)
    out["momentum"] = dlog_anom(df, "TTF").shift(horizon)
    out["storage_now"] = change_anom(df, "EU+UK_av_storage(bcm)")
    out["hdd_now"] = level_anom(df, "Europe_HDD")
    out["neasia_gwdd_now"] = level_anom(df, "NE_Asia_GWDD")
    out["storage_lag"] = change_anom(df, "EU+UK_av_storage(bcm)").shift(horizon)
    out["hdd_lag"] = level_anom(df, "Europe_HDD").shift(horizon)
    return out


def run_global_lean():
    print("\n\n" + "#" * 78)
    print("# §5.2  GLOBAL LEAN check -- isolate the one significant global channel (NE-Asia DD)")
    print("#" * 78)
    df = load_data_simple()
    feat = gl_build_features(df)

    print("GLOBAL LEAN check: does the single significant global channel (NE-Asia DD)")
    print("add OOS value on top of the lean European perfect-foresight model?\n")

    print("=" * 66)
    print("In-sample R2 build-up (train %s..%s)" % (feat["Date"].min().date(), TRAIN_END[:7]))
    print("=" * 66)
    for label, cols in [("momentum only", ["momentum"]),
                        ("+ storage (now)", ["momentum", "storage_now"]),
                        ("+ HDD (now)  [= EU LEAN]", GL_EU_LEAN),
                        ("+ NE-Asia DD [= GLOBAL LEAN]", GL_GLOBAL_LEAN),
                        ("honest EU core (all lagged)", GL_CORE_LAG)]:
        rr, n = r2(feat, cols)
        print("  %-32s R2=%.3f  (n=%d)" % (label, rr, n))

    print("\n" + "=" * 66)
    print("HAC fit -- GLOBAL LEAN (signs & significance)")
    print("=" * 66)
    m = fit_hac(feat, GL_GLOBAL_LEAN)
    print(m.summary())
    for c in GL_GLOBAL_LEAN:
        print("  %-20s coef=%+.5f  t=%+.2f  p=%.3f  sign_ok=%s"
              % (c, m.params[c], m.tvalues[c], m.pvalues[c], np.sign(m.params[c]) == GL_SIGN[c]))

    print("\n" + "=" * 66)
    print("Out-of-sample walk-forward: EU lean vs GLOBAL lean vs EU core vs RW")
    print("=" * 66)
    res = walk_forward_fixed(feat, {"core_lag": GL_CORE_LAG, "eu_lean": GL_EU_LEAN,
                                    "global_lean": GL_GLOBAL_LEAN})
    print("\n-- Full backtest (%s to %s) --" % (res["Date"].min().date(), res["Date"].max().date()))
    print(summarize(res, ["core_lag", "eu_lean", "global_lean"]).to_string(index=False))
    clean = res.loc[res["Date"] >= TRAIN_END]
    if len(clean) >= 6:
        print("\n-- Clean post-%s slice --" % TRAIN_END[:7])
        print(summarize(clean, ["core_lag", "eu_lean", "global_lean"]).to_string(index=False))
    print("\nDM t, GLOBAL lean vs EU lean (positive => Asian-pull channel helps): %.2f"
          % dm_tstat(res["actual"], res["pred_eu_lean"], res["pred_global_lean"]))
    print("DM t, GLOBAL lean vs honest EU core (positive => global helps):       %.2f"
          % dm_tstat(res["actual"], res["pred_core_lag"], res["pred_global_lean"]))

    regime_r2(feat, GL_GLOBAL_LEAN, "GLOBAL lean")
    print("\nNOTE: NE-Asia DD selected post-hoc (generous test); perfect-foresight ceiling.")


# =========================================================================== #
# §5.3  ARDL/ECM deployable + cointegration  -- ttf_ardl_ecm.py
# =========================================================================== #

In [48]:
ECM_CORE = ["momentum", "storage_change_anom", "hdd_anom"]
ECM_ECM = ECM_CORE + ["ect_lag"]


def ecm_build_features(df, horizon=HORIZON):
    out = df[["Date", "month", "log_TTF", "log_JKM", "log_HH"]].copy()
    out["y"] = df["log_TTF"].diff(horizon)
    dlog = df["log_TTF"].diff(horizon)
    out["momentum"] = (dlog - expanding_seasonal_mean(dlog, df["month"])).shift(horizon)
    stor = df["EU+UK_av_storage(bcm)"].diff()
    out["storage_change_anom"] = (stor - expanding_seasonal_mean(stor, df["month"])).shift(horizon)
    out["hdd_anom"] = (df["Europe_HDD"] - expanding_seasonal_mean(df["Europe_HDD"], df["month"])).shift(horizon)
    return out


def ecm_cointegration_report(frame, train_end=TRAIN_END):
    train = frame.loc[frame["Date"] < train_end].dropna(subset=["log_TTF"] + LONGRUN_X)
    print("=" * 78)
    print("Cointegration of log TTF with log JKM / log HH  (train %s..%s, n=%d)"
          % (frame["Date"].min().date(), train_end[:7], len(train)))
    print("=" * 78)
    lr = long_run_ols(train)
    b = lr.params
    print("Long-run relationship:  log_TTF = %.3f + %.3f*log_JKM + %.3f*log_HH"
          % (b["const"], b["log_JKM"], b["log_HH"]))
    ect = ect_from_fit(train, lr).dropna()
    adf_stat, adf_p, *_ = adfuller(ect, autolag="AIC")
    print("Engle-Granger residual ADF:  stat=%.3f  p=%.3f  -> %s"
          % (adf_stat, adf_p, "cointegrated (residual stationary)" if adf_p < 0.05
             else "NOT cointegrated at 5% (residual has a unit root)"))
    print("  (EG p-values are approximate; the residual is a regression residual, "
          "so treat ~0.05 cautiously.)")
    pre = train.loc[train["Date"] < CRISIS_START].dropna(subset=["log_TTF"] + LONGRUN_X)
    if len(pre) >= 20:
        lr_pre = long_run_ols(pre); bp = lr_pre.params
        print("\nStability across the 2022 break:")
        print("  pre-crisis (<%s, n=%d):  b_JKM=%.3f  b_HH=%.3f" % (CRISIS_START[:7], len(pre), bp["log_JKM"], bp["log_HH"]))
        print("  full train        (n=%d):  b_JKM=%.3f  b_HH=%.3f" % (len(train), b["log_JKM"], b["log_HH"]))
        ect_pre = ect_from_fit(pre, lr_pre).dropna()
        adf_pre_p = adfuller(ect_pre, autolag="AIC")[1]
        print("  pre-crisis residual ADF p=%.3f  (%s)"
              % (adf_pre_p, "stationary" if adf_pre_p < 0.05 else "not stationary"))
        print("  => %s" % ("long-run vector shifts materially across the break -- "
                           "cointegration may be unstable"
                           if abs(bp["log_JKM"] - b["log_JKM"]) > 0.25
                           or abs(bp["log_HH"] - b["log_HH"]) > 0.25
                           else "long-run vector reasonably stable across the break"))
    print("\nPesaran-Shin-Smith bounds test (level relationship):")
    try:
        from statsmodels.tsa.ardl import UECM
        uecm = UECM(train["log_TTF"], lags=2, exog=train[LONGRUN_X], order=2, trend="c").fit()
        bt = uecm.bounds_test(case=3)
        print("  F-stat = %.3f" % float(np.squeeze(bt.stat)))
        print("  critical values (10/5/1%%):\n%s" % bt.crit_vals.to_string())
        print("  => F above the I(1) upper bound => reject 'no level relationship' (cointegration).")
    except Exception as e:
        print("  [bounds test unavailable in this environment: %s]" % type(e).__name__)
        print("  Rely on the Engle-Granger result above.")
    print()

In [49]:
def ecm_insample(frame, train_end=TRAIN_END):
    train = frame.loc[frame["Date"] < train_end].copy()
    lr = long_run_ols(train.dropna(subset=["log_TTF"] + LONGRUN_X))
    ect = ect_from_fit(frame, lr)
    frame = frame.copy()
    frame["ect_lag"] = ect.shift(1)
    d = frame.loc[frame["Date"] < train_end, ["y"] + ECM_ECM].dropna()
    return sm.OLS(d["y"], sm.add_constant(d[ECM_ECM])).fit(
        cov_type="HAC", cov_kwds={"maxlags": hac_lags(len(d)), "use_correction": True})


def ecm_walk_forward(frame, min_train=ECM_MIN_TRAIN):
    work = frame.dropna(subset=["y"] + ECM_CORE + ["log_TTF"] + LONGRUN_X).reset_index(drop=True)
    rows = []
    for i in range(min_train, len(work)):
        hist = work.iloc[:i]
        lr = long_run_ols(hist)
        ect_all = ect_from_fit(work, lr)
        ect_lag = ect_all.shift(1)
        tr = hist.copy()
        tr["ect_lag"] = ect_lag.iloc[:i].values
        trd = tr[["y"] + ECM_ECM].dropna()
        if len(trd) < len(ECM_ECM) + 5:
            continue
        core_fit = sm.OLS(trd["y"], sm.add_constant(trd[ECM_CORE])).fit()
        ecm_fit = sm.OLS(trd["y"], sm.add_constant(trd[ECM_ECM])).fit()
        test = work.iloc[i]
        Xc = sm.add_constant(test[ECM_CORE].to_frame().T, has_constant="add")
        row_ecm = test[ECM_CORE].to_dict()
        row_ecm["ect_lag"] = ect_lag.iloc[i]
        Xe = sm.add_constant(pd.DataFrame([row_ecm])[ECM_ECM], has_constant="add")
        rows.append({"Date": test["Date"], "actual": test["y"],
                     "pred_random_walk": 0.0, "pred_hist_mean": hist["y"].mean(),
                     "pred_core": core_fit.predict(Xc).iloc[0],
                     "pred_ecm": ecm_fit.predict(Xe).iloc[0],
                     "alpha": ecm_fit.params["ect_lag"]})
    return pd.DataFrame(rows)

In [50]:
def run_ardl_ecm():
    print("\n\n" + "#" * 78)
    print("# §5.3  ARDL/ECM deployable + cointegration (ttf_ardl_ecm.py)")
    print("#" * 78)
    df = load_data_prices()
    feat = ecm_build_features(df, horizon=HORIZON)
    feat = feat.loc[feat["Date"] >= SAMPLE_START].reset_index(drop=True)
    print("Sample: %s to %s  (n=%d monthly obs)\n"
          % (feat["Date"].min().date(), feat["Date"].max().date(), len(feat)))

    ecm_cointegration_report(feat)

    print("=" * 78)
    print("In-sample ECM (training) -- speed of adjustment alpha on ect_lag")
    print("=" * 78)
    m = ecm_insample(feat)
    print(m.summary())
    a = m.params["ect_lag"]; ap = m.pvalues["ect_lag"]
    print("\nSpeed of adjustment alpha = %.3f (p=%.3f) -> %s" % (
        a, ap, "error-correction present (negative & significant)" if (a < 0 and ap < 0.05)
        else "no significant error correction"))

    print("\n" + "=" * 78)
    print("Out-of-sample walk-forward (H=1): European core vs ECM (core + ECT)")
    print("=" * 78)
    res = ecm_walk_forward(feat)
    print("\n-- Full backtest (%s to %s) --" % (res["Date"].min().date(), res["Date"].max().date()))
    print(summarize(res, ["core", "ecm"]).to_string(index=False))
    clean = res.loc[res["Date"] >= TRAIN_END]
    if len(clean) >= 6:
        print("\n-- Clean post-%s slice --" % TRAIN_END[:7])
        print(summarize(clean, ["core", "ecm"]).to_string(index=False))
    dm = dm_tstat(res["actual"], res["pred_core"], res["pred_ecm"])
    print("\nDM t-stat, ECM vs core (positive => ECT lowers MSE): %.2f" % dm)
    print("Mean walk-forward alpha (speed of adjustment): %.3f  (negative = error-correcting)"
          % res["alpha"].mean())


# =========================================================================== #
# §5.3  ECM ceiling / lean / regime  -- ttf_ecm_ceiling.py
# =========================================================================== #

In [51]:
CEIL_WEATHER_NOW = ["hdd_now", "cdd_now", "wind_now", "solar_now", "precip_now"]
CEIL_CORE_LAG = ["momentum", "storage_lag", "hdd_lag"]
CEIL_ECM_DEPLOY = CEIL_CORE_LAG + ["ect_lag"]
CEIL_EU_CEIL = ["momentum", "storage_now"] + CEIL_WEATHER_NOW
CEIL_ECM_CEIL = CEIL_EU_CEIL + ["ect_lag"]
CEIL_EU_LEAN = ["momentum", "storage_now", "hdd_now"]
CEIL_ECM_LEAN = CEIL_EU_LEAN + ["ect_lag"]

CEIL_SIGN = {"momentum": +1, "storage_now": -1, "storage_lag": -1,
             "hdd_now": +1, "hdd_lag": +1, "cdd_now": +1,
             "wind_now": -1, "solar_now": -1, "precip_now": -1, "ect_lag": -1}


def ceil_build_features(df, horizon=HORIZON):
    out = df[["Date", "month", "log_TTF", "log_JKM", "log_HH"]].copy()
    out["y"] = df["log_TTF"].diff(horizon)
    out["momentum"] = dlog_anom(df, "TTF").shift(horizon)
    out["storage_now"] = change_anom(df, "EU+UK_av_storage(bcm)")
    out["hdd_now"] = level_anom(df, "Europe_HDD")
    out["cdd_now"] = level_anom(df, "Europe_CDD")
    out["wind_now"] = level_anom(df, "EU_wind_speed")
    out["solar_now"] = level_anom(df, "EU_solar")
    out["precip_now"] = level_anom(df, "Nordic_precip")
    out["storage_lag"] = change_anom(df, "EU+UK_av_storage(bcm)").shift(horizon)
    out["hdd_lag"] = level_anom(df, "Europe_HDD").shift(horizon)
    return out

In [52]:
def ceil_add_static_ect(feat, train_end=TRAIN_END):
    train = feat.loc[feat["Date"] < train_end].dropna(subset=["log_TTF"] + LONGRUN_X)
    lr = long_run_ols(train)
    feat = feat.copy()
    feat["ect_lag"] = ect_from_fit(feat, lr).shift(1)
    return feat, lr


def ceil_walk_forward(feat, model_cols, min_train=MIN_TRAIN):
    base = sorted({c for cols in model_cols.values() for c in cols if c != "ect_lag"})
    need = ["Date", "y", "log_TTF"] + LONGRUN_X + base
    work = feat.dropna(subset=[c for c in need if c in feat.columns]).reset_index(drop=True)
    rows = []
    for i in range(min_train, len(work)):
        hist = work.iloc[:i]
        lr = long_run_ols(hist)
        ect_lag = ect_from_fit(work, lr).shift(1)
        w = work.copy(); w["ect_lag"] = ect_lag.values
        tr, test = w.iloc[:i], w.iloc[i]
        rec = {"Date": test["Date"], "actual": test["y"],
               "pred_random_walk": 0.0, "pred_hist_mean": tr["y"].mean()}
        for name, cols in model_cols.items():
            d = tr[["y"] + cols].dropna()
            if len(d) < len(cols) + 5:
                rec[f"pred_{name}"] = np.nan
                continue
            fit = sm.OLS(d["y"], sm.add_constant(d[cols])).fit()
            Xte = sm.add_constant(test[cols].to_frame().T, has_constant="add")
            rec[f"pred_{name}"] = fit.predict(Xte).iloc[0]
        rows.append(rec)
    return pd.DataFrame(rows)

In [53]:
def run_ecm_ceiling():
    print("\n\n" + "#" * 78)
    print("# §5.3  ECM perfect-foresight CEILING + lean/regime (ttf_ecm_ceiling.py)")
    print("#" * 78)
    df = load_data_prices()
    feat = ceil_build_features(df)
    feat = feat.loc[feat["Date"] >= SAMPLE_START].reset_index(drop=True)
    feat, lr = ceil_add_static_ect(feat)
    b = lr.params
    print("ECM perfect-foresight CEILING + lean/regime.  Sample %s..%s (n=%d).\n"
          % (feat["Date"].min().date(), feat["Date"].max().date(), len(feat)))
    print("Training long-run vector: log_TTF = %.3f + %.3f*log_JKM + %.3f*log_HH"
          % (b["const"], b["log_JKM"], b["log_HH"]))
    print("Foresight granted to FUNDAMENTALS only (storage + weather); momentum & ECT lagged.\n")

    print("=" * 72)
    print("In-sample R2 build-up (train %s..%s)" % (feat["Date"].min().date(), TRAIN_END[:7]))
    print("=" * 72)
    decomp = [
        ("momentum only (lagged)",                 ["momentum"]),
        ("EU fundamentals ceiling (foresight)",    CEIL_EU_CEIL),
        ("+ ect_lag  [= ECM ceiling]",             CEIL_ECM_CEIL),
        ("(ref) honest deployable ECM (all lag)",  CEIL_ECM_DEPLOY),
        ("(ref) honest EU core (all lagged)",      CEIL_CORE_LAG),
    ]
    for label, cols in decomp:
        rr, n = r2(feat, cols)
        print("  %-38s R2=%.3f  (n=%d)" % (label, rr, n))
    print("\n  Read-off: EU ceiling is the established ~0.21 fundamentals ceiling. The ect_lag")
    print("  increment is the error-correction channel's marginal in-sample contribution on")
    print("  top of perfect foresight of the fundamentals.")

    print("\n" + "=" * 72)
    print("HAC fit -- ECM CEILING (foresight fundamentals + lagged ECT): signs?")
    print("=" * 72)
    m = fit_hac(feat, CEIL_ECM_CEIL)
    print(m.summary())
    print("\nSign check (coef sign vs economic prior):")
    for c in CEIL_ECM_CEIL:
        ok = np.sign(m.params[c]) == CEIL_SIGN[c]
        print("  %-14s coef=%+.5f  t=%+.2f  p=%.3f  sign_ok=%s"
              % (c, m.params[c], m.tvalues[c], m.pvalues[c], ok))
    print("  (ect_lag < 0 & significant => error-correction adds forecast signal.)")

    print("\n" + "=" * 72)
    print("Out-of-sample walk-forward (same window for all models, min_train=%d)" % MIN_TRAIN)
    print("=" * 72)
    res = ceil_walk_forward(feat, {"core_lag": CEIL_CORE_LAG, "ecm_deploy": CEIL_ECM_DEPLOY,
                                   "eu_ceiling": CEIL_EU_CEIL, "ecm_ceiling": CEIL_ECM_CEIL})
    names = ["core_lag", "ecm_deploy", "eu_ceiling", "ecm_ceiling"]
    print("\n-- Full backtest (%s to %s) --" % (res["Date"].min().date(), res["Date"].max().date()))
    print(summarize(res, names).to_string(index=False))
    clean = res.loc[res["Date"] >= TRAIN_END]
    if len(clean) >= 6:
        print("\n-- Clean post-%s slice --" % TRAIN_END[:7])
        print(summarize(clean, names).to_string(index=False))
    print("\nDM t, ECM ceiling vs EU fundamentals ceiling (positive => ECT helps): %.2f"
          % dm_tstat(res["actual"], res["pred_eu_ceiling"], res["pred_ecm_ceiling"]))
    print("DM t, ECM ceiling vs honest EU core (positive => ceiling+ECT helps):   %.2f"
          % dm_tstat(res["actual"], res["pred_core_lag"], res["pred_ecm_ceiling"]))
    print("DM t, deployable ECM vs honest EU core (positive => ECT helps deployably): %.2f"
          % dm_tstat(res["actual"], res["pred_core_lag"], res["pred_ecm_deploy"]))

    print("\n" + "=" * 72)
    print("LEAN cut: EU lean vs ECM lean (does ECT help a tight 4-param model?)")
    print("=" * 72)
    resl = ceil_walk_forward(feat, {"core_lag": CEIL_CORE_LAG, "eu_lean": CEIL_EU_LEAN,
                                    "ecm_lean": CEIL_ECM_LEAN})
    lnames = ["core_lag", "eu_lean", "ecm_lean"]
    print("\n-- Full backtest --")
    print(summarize(resl, lnames).to_string(index=False))
    cleanl = resl.loc[resl["Date"] >= TRAIN_END]
    if len(cleanl) >= 6:
        print("\n-- Clean post-%s slice --" % TRAIN_END[:7])
        print(summarize(cleanl, lnames).to_string(index=False))
    print("\nDM t, ECM lean vs EU lean (positive => ECT helps):   %.2f"
          % dm_tstat(resl["actual"], resl["pred_eu_lean"], resl["pred_ecm_lean"]))
    mlean = fit_hac(feat, CEIL_ECM_LEAN)
    print("\nECM lean HAC signs:")
    for c in CEIL_ECM_LEAN:
        print("  %-14s coef=%+.5f  t=%+.2f  p=%.3f  sign_ok=%s"
              % (c, mlean.params[c], mlean.tvalues[c], mlean.pvalues[c], np.sign(mlean.params[c]) == CEIL_SIGN[c]))

    regime_r2(feat, CEIL_ECM_CEIL, "ECM ceiling")
    regime_r2(feat, CEIL_ECM_LEAN, "ECM lean")
    print("\nNOTE: perfect-foresight ceiling on FUNDAMENTALS; ECT lagged (honest); prices not foreseen.")


# =========================================================================== #
# §5.4  VECM transmission  -- ttf_vecm.py (parametrized by Cholesky ordering)
# =========================================================================== #

In [54]:
VECM_LABEL = {"log_HH": "HenryHub", "log_JKM": "JKM", "log_TTF": "TTF"}
ORDER_HEADLINE = ["log_TTF", "log_JKM", "log_HH"]   # data-driven (TTF leads) -> reported §5.4
ORDER_ROBUST = ["log_HH", "log_JKM", "log_TTF"]     # a-priori (US insulated) -> robustness / §3.3
VECM_MAXLAGS = 8
VECM_DET_ORDER = 0
VECM_IRF_H = 18
VECM_FEVD_HORIZONS = [1, 3, 6, 12, 18]


def vecm_load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    jkm = df["JKM(USD/mmbtu)"].astype(float).copy()
    jkm[np.isclose(jkm, 7.86)] = np.nan
    df["JKM(USD/mmbtu)"] = np.exp(np.log(jkm).interpolate(limit_direction="both"))
    df["log_TTF"] = np.log(df["TTF(USD/mmbtu)"])
    df["log_JKM"] = np.log(df["JKM(USD/mmbtu)"])
    df["log_HH"] = np.log(df["HH(USD/mmbtu)"])
    df = df.loc[df["Date"] >= SAMPLE_START].reset_index(drop=True)
    return df

In [55]:
def vecm_adf_line(series, name):
    s = series.dropna()
    stat, p, *_ = adfuller(s, autolag="AIC")
    return "  %-16s ADF stat=%+.3f  p=%.3f  -> %s" % (
        name, stat, p, "stationary" if p < 0.05 else "unit root (non-stationary)")


def run_vecm(order, tag, save_suffix):
    print("\n\n" + "#" * 78)
    print("# §5.4  VECM transmission -- %s" % tag)
    print("#" * 78)
    df = vecm_load_data()
    data = df[order].copy()
    K = len(order)
    ttf_i = order.index("log_TTF")
    print("VECM transmission study -- system order (Cholesky): %s"
          % " -> ".join(VECM_LABEL[c] for c in order))
    print("Sample %s..%s  (n=%d monthly obs)\n" % (df["Date"].min().date(), df["Date"].max().date(), len(df)))

    print("=" * 74)
    print("1. Integration order (ADF)")
    print("=" * 74)
    print("Levels (expect unit root):")
    for c in order:
        print(vecm_adf_line(data[c], VECM_LABEL[c]))
    print("First differences (expect stationary):")
    for c in order:
        print(vecm_adf_line(data[c].diff(), "d." + VECM_LABEL[c]))

    print("\n" + "=" * 74)
    print("2. Lag-order selection (levels VAR, maxlags=%d)" % VECM_MAXLAGS)
    print("=" * 74)
    lo = select_order(data, maxlags=VECM_MAXLAGS, deterministic="ci")
    print(lo.summary())
    p_levels = max(int(lo.aic), 2)
    k_ar_diff = p_levels - 1
    print("\nChosen levels-VAR lag p = %d  (VECM k_ar_diff = %d)  [AIC]" % (p_levels, k_ar_diff))

    print("\n" + "=" * 74)
    print("3. Johansen cointegration rank (det_order=%d = restricted constant)" % VECM_DET_ORDER)
    print("=" * 74)
    joh = coint_johansen(data, VECM_DET_ORDER, k_ar_diff)
    print("  Trace test:                stat      cv90      cv95      cv99")
    for i in range(K):
        print("   r <= %d :   %10.3f %9.3f %9.3f %9.3f"
              % (i, joh.lr1[i], joh.cvt[i, 0], joh.cvt[i, 1], joh.cvt[i, 2]))
    print("  Max-eigenvalue test:       stat      cv90      cv95      cv99")
    for i in range(K):
        print("   r  = %d :   %10.3f %9.3f %9.3f %9.3f"
              % (i, joh.lr2[i], joh.cvm[i, 0], joh.cvm[i, 1], joh.cvm[i, 2]))
    rank_sel = select_coint_rank(data, VECM_DET_ORDER, k_ar_diff, method="trace", signif=0.05)
    r = max(int(rank_sel.rank), 1)
    print("\n  select_coint_rank (trace, 5%%) => rank r = %d" % rank_sel.rank)
    print("  (Using r = %d for the VECM. r=1 => a single long-run equilibrium.)" % r)

    print("\n" + "=" * 74)
    print("4. VECM estimates -- equilibrium & who adjusts")
    print("=" * 74)
    vecm = VECM(data, k_ar_diff=k_ar_diff, coint_rank=r, deterministic="ci")
    res = vecm.fit()
    print(res.summary())

    beta = np.asarray(res.beta).reshape(K, r)
    print("\nCointegrating relation(s) (rank r=%d; each normalized on its leading variable):" % r)
    for k in range(r):
        bk = beta[:, k]; lead = k
        rhs = " ".join("%+.3f*log_%s" % (-bk[j] / bk[lead], VECM_LABEL[c])
                       for j, c in enumerate(order) if j != lead)
        print("   [%d] log_%-9s = %s   (+ const)" % (k + 1, VECM_LABEL[order[lead]], rhs))
    print("   (compare ARDL/ECM single relation: log_TTF = -0.409 + 1.053 log_JKM + 0.151 log_HH)")

    alpha = np.asarray(res.alpha).reshape(K, r)
    pa = np.asarray(res.pvalues_alpha).reshape(K, r)
    print("\nAdjustment loadings alpha (per equation, across all %d relation(s)):" % r)
    print("A variable is weakly exogenous (a DRIVER) only if ALL its loadings are ~0.")
    for i, c in enumerate(order):
        cells, anysig, best = [], False, None
        for k in range(r):
            sig = pa[i, k] < 0.05
            anysig = anysig or sig
            cells.append("ec%d a=%+.4f p=%.3f%s" % (k + 1, alpha[i, k], pa[i, k], "*" if sig else " "))
            if sig and (best is None or pa[i, k] < pa[i, best]):
                best = k
        role = "error-corrects (ADJUSTER)" if anysig else "weakly exogenous (DRIVER)"
        hl = ""
        if best is not None and -1 < alpha[i, best] < 0:
            hl = "  half-life~%.1f mo" % (np.log(0.5) / np.log(1 + alpha[i, best]))
        print("   d.%-9s  %s  -> %s%s" % (VECM_LABEL[c], " | ".join(cells), role, hl))

    print("\n" + "=" * 74)
    print("5. Levels VAR (lag %d) -- Granger causality, IRF, FEVD" % p_levels)
    print("=" * 74)
    var_res = VAR(data).fit(p_levels)
    print("\nGranger causality (F-test, does past X help predict Y):")
    for causing in order:
        for caused in order:
            if causing == caused:
                continue
            t = var_res.test_causality(caused, [causing], kind="f")
            flag = "YES" if t.pvalue < 0.05 else "no "
            print("   %-9s -> %-9s  F=%7.2f  p=%.3f  [%s]"
                  % (VECM_LABEL[causing], VECM_LABEL[caused], t.test_statistic, t.pvalue, flag))

    irf = var_res.irf(VECM_IRF_H)
    cum = np.cumsum(np.asarray(irf.orth_irfs), axis=0)
    print("\nCumulative orthogonalized response of TTF to a 1-SD shock (Cholesky %s):"
          % " -> ".join(VECM_LABEL[c] for c in order))
    print("   horizon " + "".join("  %10s" % ("<-" + VECM_LABEL[c]) for c in order))
    for h in VECM_FEVD_HORIZONS:
        print("   %6d  " % h + "".join("  %10.4f" % cum[h, ttf_i, s] for s in range(K)))
    print("   (columns = shock origin; response = cumulative change in log TTF)")

    print("\nResponse of the OTHER hubs to a TTF shock (does Europe feed back?):")
    for resp in order:
        if resp == "log_TTF":
            continue
        ri = order.index(resp)
        vals = "  ".join("h%d=%+.4f" % (h, cum[h, ri, ttf_i]) for h in [3, 6, 12])
        print("   TTF-shock -> %-9s : %s" % (VECM_LABEL[resp], vals))

    fevd = var_res.fevd(max(VECM_FEVD_HORIZONS))
    dec = np.asarray(fevd.decomp)
    print("\nFEVD of TTF -- share of its forecast-error variance from each hub's shocks:")
    print("   horizon " + "".join("  %8s" % VECM_LABEL[c] for c in order))
    for h in VECM_FEVD_HORIZONS:
        print("   %6d " % h + "".join("  %7.1f%%" % (100 * dec[ttf_i, h - 1, s]) for s in range(K)))

    print("\n" + "=" * 74)
    print("Transmission read-off")
    print("=" * 74)
    print("  - rank r and beta give the long-run equilibrium; compare to the ECM vector.")
    print("  - alpha shows who adjusts: an equation with a significant loading error-corrects;")
    print("    a variable whose loadings are ALL ~0 is weakly exogenous (a DRIVER). This is the")
    print("    transmission structure the single-equation ECM assumed but could not test.")
    print("  - IRF/FEVD quantify how much, and how fast, JKM/HH shocks move TTF. Multi-MONTH")
    print("    structure, consistent with the ECM finding that adjustment is too slow at H=1.")

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt  # noqa: F401
        fig = irf.plot(orth=True, response="log_TTF")
        fig.suptitle("Orthogonalized IRF: response of log TTF (%s)" % save_suffix)
        fig.savefig("vecm_irf_TTF_%s.png" % save_suffix, dpi=120, bbox_inches="tight")
        fig2 = fevd.plot()
        fig2.savefig("vecm_fevd_%s.png" % save_suffix, dpi=120, bbox_inches="tight")
        print("\n[saved vecm_irf_TTF_%s.png and vecm_fevd_%s.png]" % (save_suffix, save_suffix))
    except Exception as e:
        print("\n[plots skipped: %s]" % type(e).__name__)


# ttf_vecm_split.py -- regime split of the VECM transmission study. Re-estimates
# rank / adjustment / Granger / FEVD on pre-crisis (<2021-09) and crisis-on
# (>=2021-09) sub-samples and prints a leadership scorecard. Reuses vecm_load_data
# (identical JKM-interp + log-price loader); helpers prefixed vs_ to avoid clashes.

In [56]:
VS_BREAK = "2021-09-01"                              # crisis onset (== CRISIS_START)
VS_K_AR_DIFF = 1                                     # fixed across sub-samples (levels-VAR p=2)
VS_P_LEVELS = 2
VS_DET_ORDER = 0                                     # restricted constant (matches VECM "ci")
VS_FEVD_H = 12
VS_BASE_ORDER = ["log_TTF", "log_JKM", "log_HH"]
VS_LABEL = {"log_TTF": "TTF", "log_JKM": "JKM", "log_HH": "HenryHub"}


def vs_half_life(a):
    return np.log(0.5) / np.log(1 + a) if -1 < a < 0 else np.nan


def vs_analyze(sub, label):
    K = len(VS_BASE_ORDER)
    data = sub[VS_BASE_ORDER].copy()
    print("\n" + "#" * 74)
    print("# %s   (%s to %s, n=%d)" % (label, sub["Date"].min().date(), sub["Date"].max().date(), len(data)))
    print("#" * 74)
    metrics = {"label": label, "n": len(data)}
    # 1. Johansen rank
    joh = coint_johansen(data, VS_DET_ORDER, VS_K_AR_DIFF)
    print("Johansen trace:   " + "  ".join("r<=%d: %.1f (cv95 %.1f)" % (i, joh.lr1[i], joh.cvt[i, 1]) for i in range(K)))
    try:
        r_sel = int(select_coint_rank(data, VS_DET_ORDER, VS_K_AR_DIFF, method="trace", signif=0.05).rank)
    except Exception:
        r_sel = int(np.sum(joh.lr1 > joh.cvt[:, 1]))
    r = max(r_sel, 1)
    metrics["rank"] = r_sel
    print("  => selected cointegration rank r = %d  (using r=%d for the VECM)" % (r_sel, r))
    # 2. VECM adjustment / weak exogeneity
    driver = None
    try:
        res = VECM(data, k_ar_diff=VS_K_AR_DIFF, coint_rank=r, deterministic="ci").fit()
        alpha = np.asarray(res.alpha).reshape(K, r)
        pa = np.asarray(res.pvalues_alpha).reshape(K, r)
        print("Adjustment loadings alpha (per equation; weakly exogenous = all loadings ~0):")
        weakly_exog = []
        for i, c in enumerate(VS_BASE_ORDER):
            cells, anysig, best = [], False, None
            for k in range(r):
                sig = pa[i, k] < 0.05
                anysig = anysig or sig
                cells.append("ec%d a=%+.3f p=%.3f%s" % (k + 1, alpha[i, k], pa[i, k], "*" if sig else " "))
                if sig and (best is None or pa[i, k] < pa[i, best]):
                    best = k
            role = "ADJUSTER" if anysig else "DRIVER (weakly exog)"
            hl = ("  half-life~%.1fmo" % vs_half_life(alpha[i, best])) if best is not None else ""
            print("   d.%-9s %s -> %s%s" % (VS_LABEL[c], " | ".join(cells), role, hl))
            if not anysig:
                weakly_exog.append(c)
            metrics["%s_adjusts" % VS_LABEL[c]] = anysig
        if len(weakly_exog) == 1:
            driver = weakly_exog[0]
        metrics["driver"] = VS_LABEL[driver] if driver else ("+".join(VS_LABEL[c] for c in weakly_exog) if weakly_exog else "none/ambiguous")
    except Exception as e:
        print("  [VECM fit failed: %s -- reporting Granger only]" % type(e).__name__)
        metrics["driver"] = "n/a"
    # 3. Granger causality (levels VAR)
    print("Granger causality (F-test p; significant at 5%% flagged):")
    var_res = VAR(data).fit(VS_P_LEVELS)
    for causing in VS_BASE_ORDER:
        line = []
        for caused in VS_BASE_ORDER:
            if causing == caused:
                continue
            t = var_res.test_causality(caused, [causing], kind="f")
            line.append("%s->%s p=%.3f%s" % (VS_LABEL[causing], VS_LABEL[caused], t.pvalue, "*" if t.pvalue < 0.05 else ""))
            metrics["G_%s_%s" % (VS_LABEL[causing], VS_LABEL[caused])] = t.pvalue
        print("   " + "   ".join(line))
    # 4. FEVD own-shares under a leader-first ordering
    lead = driver if driver else "log_TTF"
    order = [lead] + [c for c in VS_BASE_ORDER if c != lead]
    vr = VAR(sub[order].copy()).fit(VS_P_LEVELS)
    dec = np.asarray(vr.fevd(VS_FEVD_H).decomp)      # [var, horizon, shock]
    print("FEVD own-variance share at h=%d (ordering %s, leader first):"
          % (VS_FEVD_H, "->".join(VS_LABEL[c] for c in order)))
    for i, c in enumerate(order):
        own = dec[i, VS_FEVD_H - 1, i]
        print("   %-9s own-share = %5.1f%%" % (VS_LABEL[c], 100 * own))
        metrics["%s_own_fevd" % VS_LABEL[c]] = own
    return metrics

In [57]:
def vs_scorecard(pre, post):
    print("\n" + "=" * 74)
    print("LEADERSHIP SCORECARD  (did the leader flip pre- vs post-2021?)")
    print("=" * 74)
    rows = [
        ("Cointegration rank (trace)",      "rank"),
        ("TTF is DRIVER (weakly exog)?",    "_ttf_driver"),
        ("TTF adjusts?",                    "TTF_adjusts"),
        ("JKM adjusts?",                    "JKM_adjusts"),
        ("HenryHub adjusts?",               "HenryHub_adjusts"),
        ("Granger TTF->JKM  (p)",           "G_TTF_JKM"),
        ("Granger JKM->TTF  (p)",           "G_JKM_TTF"),
        ("Granger TTF->HenryHub (p)",       "G_TTF_HenryHub"),
        ("Granger HenryHub->TTF (p)",       "G_HenryHub_TTF"),
        ("TTF own-FEVD share (h=%d)" % VS_FEVD_H, "TTF_own_fevd"),
    ]
    pre["_ttf_driver"] = (pre.get("driver") == "TTF")
    post["_ttf_driver"] = (post.get("driver") == "TTF")
    print("  %-32s %14s %14s" % ("", pre["label"], post["label"]))
    for name, key in rows:
        def fmtval(m):
            v = m.get(key, np.nan)
            if isinstance(v, (bool, np.bool_)):
                return "yes" if v else "no"
            if key == "TTF_own_fevd":
                return "n/a" if (v is None or v != v) else "%.1f%%" % (100 * v)
            if isinstance(v, (float, np.floating)):
                return "n/a" if v != v else ("%.3f" % v)
            return str(v)
        print("  %-32s %14s %14s" % (name, fmtval(pre), fmtval(post)))
    print("\n  Driver detected:   %-14s -> %s" % (pre.get("driver"), post.get("driver")))
    print("\nRead: if TTF is an ADJUSTER (and JKM the driver / Granger-leader) pre-crisis but")
    print("the DRIVER (Granger-leading JKM) crisis-on, leadership flipped from Asia to Europe.")

In [58]:
def run_vecm_regime_split():
    print("\n\n" + "#" * 78)
    print("# §5.4  VECM regime split (pre/post 2021-09)  [ttf_vecm_split.py]")
    print("#" * 78)
    df = vecm_load_data()                            # identical loader (JKM interp + log prices)
    print("VECM regime split -- pre/post %s.  Full sample %s..%s (n=%d)."
          % (VS_BREAK[:7], df["Date"].min().date(), df["Date"].max().date(), len(df)))
    print("Fixed k_ar_diff=%d for both. Johansen CVs asymptotic; small-n => indicative.\n" % VS_K_AR_DIFF)
    pre = df.loc[df["Date"] < VS_BREAK].reset_index(drop=True)
    post = df.loc[df["Date"] >= VS_BREAK].reset_index(drop=True)
    m_pre = vs_analyze(pre, "PRE-CRISIS  (< %s)" % VS_BREAK[:7])
    m_post = vs_analyze(post, "CRISIS-ON   (>= %s)" % VS_BREAK[:7])
    vs_scorecard(m_pre, m_post)


# =========================================================================== #
# Master entry point -- runs every available global sub-model in paper order
# =========================================================================== #

def main():
    run_global_compact()                                   # §5.1  (stub -- script not supplied)
    run_global_conditional()                               # §5.2  ceiling
    run_global_lean()                                      # §5.2  lean / NE-Asia isolation
    run_ardl_ecm()                                         # §5.3  deployable ECM + cointegration
    run_ecm_ceiling()                                      # §5.3  ECM ceiling / lean / regime
    run_vecm(ORDER_HEADLINE, "HEADLINE (data-driven ordering, TTF first)", "headline")   # §5.4
    run_vecm(ORDER_ROBUST, "ROBUSTNESS (a-priori ordering, HH first)", "robust")          # §5.4
    run_vecm_regime_split()                                # §5.4  regime split (pre/post 2021-09)

In [59]:
# ---- run this section ----
main()



##############################################################################
# §5.1  GLOBAL COMPACT (deployable, all-lagged pool)  [ttf_global_model.py]
##############################################################################
Candidate pool: 55 total  (35 European, 19 Global, 1 Own/momentum)

Stationarity gate (training sample, H=1 candidates)
                        name        adf_p   kpss_p        verdict  usable
                    momentum 2.076585e-12 0.100000     stationary    True
         storage_change_anom 1.129799e-06 0.100000     stationary    True
          storage_level_anom 1.129812e-01 0.100000    conflicting    True
             production_anom 5.398473e-01 0.010000 non-stationary   False
              net_piped_anom 7.866081e-01 0.010000 non-stationary   False
            lng_imports_anom 1.456683e-01 0.010000 non-stationary   False
             net_supply_anom 7.789721e-01 0.010000 non-stationary   False
           total_demand_anom 1.750841e-01 0.010000 n

---

## Section 6 — Volatility (ARCH / GARCH)

Source: `Global_gas_volatility.py`

<details><summary>original module docstring</summary>

```
ttf_garch.py

Phase 2, model (5): ARCH/GARCH volatility study for European gas (TTF).

Every earlier study in this series found the same thing about the CONDITIONAL
MEAN of month-ahead TTF: it is close to a random walk, with what little
explanatory power there is concentrated in high-volatility / crisis months and
near-absent in calm ones. This study turns to the CONDITIONAL VARIANCE, which is
where that regime structure actually lives. It asks:

  1. Is there an ARCH effect to model at all (ARCH-LM test)?
  2. Which volatility specification fits best -- ARCH(1), GARCH(1,1), GJR-GARCH
     (asymmetry / leverage), EGARCH -- and under Normal vs Student-t errors
     (gas returns are fat-tailed)?
  3. How PERSISTENT is volatility (alpha+beta and the shock half-life)? Gas vol
     is famously near-integrated.
  4. Is the asymmetry the "inverse leverage" of commodities -- do POSITIVE price
     shocks (supply scares) raise volatility more than negative ones (unlike
     equities)? (sign of the GJR/EGARCH asymmetry term.)
  5. Do fundamentals that were useless for the MEAN (storage, HDD) help explain
     the VARIANCE? (post-hoc regression of conditional vol on fundamental stress.)
  6. Does the GARCH model actually FORECAST variance better than simple
     benchmarks out of sample -- EWMA (RiskMetrics) and a rolling window --
     judged by QLIKE and MSE loss against realized squared returns, with a
     Diebold-Mariano test? (the same OOS discipline used throughout.)

Returns are monthly log returns of TTF x100 (percent), as the arch optimizer
prefers percent-scale data. Monthly GARCH on ~136 returns is data-hungry, so
read magnitudes as indicative and lean on the qualitative structure.

Requirements: arch (pip install arch), statsmodels, numpy, pandas, scipy
```

</details>

In [60]:
import numpy as np
import pandas as pd
from arch import arch_model
from statsmodels.stats.diagnostic import het_arch, acorr_ljungbox
import statsmodels.api as sm

DATA_PATH = "NG_m_final.csv"
SAMPLE_START = "2015-01-01"
CRISIS_START, CRISIS_END = "2021-09-01", "2023-07-01"
OOS_MIN_TRAIN = 72        # 6 yrs before the first OOS variance forecast
LB_LAGS = 12
EWMA_LAMBDA = 0.94        # RiskMetrics; also report a rolling-window benchmark
ROLL_WIN = 12

In [61]:
# --------------------------------------------------------------------------- #
# Data
# --------------------------------------------------------------------------- #

def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df["month"] = df["Date"].dt.month
    df = df.loc[df["Date"] >= SAMPLE_START].reset_index(drop=True)
    return df


def expanding_seasonal_mean(series, month):
    out = pd.Series(index=series.index, dtype=float)
    tmp = pd.DataFrame({"val": series, "month": month})
    for _, grp in tmp.groupby("month"):
        out.loc[grp.index] = grp["val"].expanding().mean().shift(1)
    return out


def build_returns(df):
    """Monthly log return of TTF in percent, plus aligned fundamental stress."""
    r = 100.0 * np.log(df["TTF(USD/mmbtu)"]).diff()
    stor_chg = df["EU+UK_av_storage(bcm)"].diff()
    stor_anom = stor_chg - expanding_seasonal_mean(stor_chg, df["month"])
    hdd_anom = df["Europe_HDD"] - expanding_seasonal_mean(df["Europe_HDD"], df["month"])
    out = pd.DataFrame({
        "Date": df["Date"], "r": r,
        "abs_stor_anom": stor_anom.abs(),
        "abs_hdd_anom": hdd_anom.abs(),
    }).iloc[1:].reset_index(drop=True)      # drop first (NaN return)
    return out

In [62]:
# --------------------------------------------------------------------------- #
# Spec helpers
# --------------------------------------------------------------------------- #

SPECS = [
    ("ARCH(1)-N",      dict(vol="ARCH",   p=1, o=0, q=0, dist="normal")),
    ("GARCH(1,1)-N",   dict(vol="GARCH",  p=1, o=0, q=1, dist="normal")),
    ("GARCH(1,1)-t",   dict(vol="GARCH",  p=1, o=0, q=1, dist="t")),
    ("GJR(1,1)-t",     dict(vol="GARCH",  p=1, o=1, q=1, dist="t")),
    ("EGARCH(1,1)-t",  dict(vol="EGARCH", p=1, o=1, q=1, dist="t")),
]


def fit_spec(r, vol="GARCH", p=1, o=0, q=1, dist="normal", mean="Constant"):
    am = arch_model(r, mean=mean, vol=vol, p=p, o=o, q=q, dist=dist)
    return am.fit(disp="off")


def persistence(res, vol, p=1, o=0, q=1):
    pr = res.params
    b = sum(pr.get("beta[%d]" % i, 0.0) for i in range(1, q + 1))
    if vol == "EGARCH":
        return b                                        # beta is the persistence in EGARCH
    a = sum(pr.get("alpha[%d]" % i, 0.0) for i in range(1, p + 1))
    g = sum(pr.get("gamma[%d]" % i, 0.0) for i in range(1, o + 1))
    return a + b + g / 2.0                               # +gamma/2: shock is negative w.p. 0.5


def half_life(pers):
    return np.log(0.5) / np.log(pers) if 0 < pers < 1 else np.inf

In [63]:
# --------------------------------------------------------------------------- #
# OOS variance-forecast benchmarks + losses
# --------------------------------------------------------------------------- #

def ewma_pred_var(r, lam=EWMA_LAMBDA, init_win=24):
    """One-step-ahead EWMA variance: pred[t] uses info through t-1."""
    r = np.asarray(r, float)
    n = len(r)
    pred = np.full(n, np.nan)
    s2 = np.nanvar(r[:init_win])
    for t in range(1, n):
        s2 = lam * s2 + (1 - lam) * r[t - 1] ** 2
        pred[t] = s2
    return pred


def rolling_pred_var(r, win=ROLL_WIN):
    r = pd.Series(np.asarray(r, float))
    return r.rolling(win).var().shift(1).values     # pred[t] = var of r[t-win..t-1]


def qlike(realized, pred):
    m = (~np.isnan(pred)) & (pred > 0)
    rz, pz = realized[m], pred[m]
    return np.mean(np.log(pz) + rz / pz)            # realized = r^2 here


def mse(realized, pred):
    m = ~np.isnan(pred)
    return np.mean((pred[m] - realized[m]) ** 2)


def dm_tstat(loss_worse, loss_better):
    d = np.asarray(loss_worse, float) - np.asarray(loss_better, float)
    d = d[~np.isnan(d)]
    n = len(d)
    return d.mean() / (d.std(ddof=1) / np.sqrt(n)) if d.std(ddof=1) > 0 else np.nan

In [64]:
# --------------------------------------------------------------------------- #
# Main
# --------------------------------------------------------------------------- #

def main():
    df = load_data()
    dat = build_returns(df)
    r = dat["r"]
    print("TTF GARCH volatility study.  Sample %s..%s  (%d monthly returns, percent).\n"
          % (dat["Date"].min().date(), dat["Date"].max().date(), len(r)))

    # ---- 1. ARCH-LM test: is there volatility clustering to model? ----
    print("=" * 70)
    print("1. ARCH-LM test on returns (H0: no ARCH effect)")
    print("=" * 70)
    lm, lmp, f, fp = het_arch(r - r.mean(), nlags=LB_LAGS)
    print("   LM stat=%.2f  p=%.4f   F=%.2f  p=%.4f  -> %s"
          % (lm, lmp, f, fp, "ARCH effect present (GARCH warranted)" if lmp < 0.05
             else "no significant ARCH effect"))

    # ---- 2. Model comparison ----
    print("\n" + "=" * 70)
    print("2. Volatility-model comparison (mean=Constant)")
    print("=" * 70)
    print("   %-15s %9s %9s %9s %11s %9s" % ("Spec", "logL", "AIC", "BIC", "persist", "half-life"))
    fits = {}
    for name, kw in SPECS:
        try:
            res = fit_spec(r, **kw)
            fits[name] = (res, kw)
            pers = persistence(res, kw["vol"], kw["p"], kw["o"], kw["q"])
            hl = half_life(pers)
            print("   %-15s %9.2f %9.2f %9.2f %11.3f %9s"
                  % (name, res.loglikelihood, res.aic, res.bic, pers,
                     ("%.1f mo" % hl) if np.isfinite(hl) else "inf"))
        except Exception as e:
            print("   %-15s  [fit failed: %s]" % (name, type(e).__name__))
    best_name = min(fits, key=lambda k: fits[k][0].bic)
    print("\n   Best by BIC: %s" % best_name)

    # ---- 3. Best model: parameters, asymmetry, residual diagnostics ----
    best, bkw = fits[best_name]
    print("\n" + "=" * 70)
    print("3. Best model (%s) -- parameters & diagnostics" % best_name)
    print("=" * 70)
    print(best.summary())
    # asymmetry sign (if present): gamma
    if "gamma[1]" in best.params.index:
        g = best.params["gamma[1]"]; gp = best.pvalues["gamma[1]"]
        if bkw["vol"] == "EGARCH":
            tell = ("positive shocks raise vol MORE (inverse leverage, commodity-like)"
                    if g > 0 else "negative shocks raise vol more (equity-like leverage)")
        else:  # GJR: gamma multiplies negative-shock term; gamma<0 => positive shocks raise vol more
            tell = ("negative shocks raise vol more (equity-like leverage)"
                    if g > 0 else "positive shocks raise vol MORE (inverse leverage, commodity-like)")
        print("\n   Asymmetry gamma=%+.4f (p=%.3f) -> %s" % (g, gp, tell))
    # standardized-residual adequacy
    z = pd.Series(np.asarray(best.std_resid, float)).dropna()
    lb = acorr_ljungbox(z, lags=[LB_LAGS], return_df=True)
    lb2 = acorr_ljungbox(z ** 2, lags=[LB_LAGS], return_df=True)
    arch_after = het_arch(z, nlags=LB_LAGS)
    print("\n   Standardized-residual diagnostics (want all INSIGNIFICANT):")
    print("     Ljung-Box(z, %d):   stat=%.2f  p=%.3f" % (LB_LAGS, lb["lb_stat"].iloc[0], lb["lb_pvalue"].iloc[0]))
    print("     Ljung-Box(z^2, %d): stat=%.2f  p=%.3f" % (LB_LAGS, lb2["lb_stat"].iloc[0], lb2["lb_pvalue"].iloc[0]))
    print("     ARCH-LM(z, %d):     p=%.3f  -> %s" % (LB_LAGS, arch_after[1],
          "clustering absorbed" if arch_after[1] >= 0.05 else "residual ARCH remains"))

    # ---- 4. Conditional volatility: level and regimes ----
    cv = np.asarray(best.conditional_volatility, float)      # monthly %, same length as r
    dat = dat.copy(); dat["cv"] = cv
    ann = cv * np.sqrt(12)
    peak_i = int(np.nanargmax(cv))
    print("\n" + "=" * 70)
    print("4. Conditional volatility (annualized = monthly x sqrt(12))")
    print("=" * 70)
    print("   mean annualized vol: %.1f%%   min: %.1f%%   peak: %.1f%% (%s)"
          % (np.nanmean(ann), np.nanmin(ann), np.nanmax(ann), dat["Date"].iloc[peak_i].date()))
    dd = dat.copy()
    dd["crisis"] = ((dd["Date"] >= CRISIS_START) & (dd["Date"] < CRISIS_END)).astype(int)
    vol6 = dd["r"].rolling(6).std().shift(1)
    dd["highvol"] = (vol6 > vol6.expanding().median().shift(1)).astype(float)
    for col, hi, lo in [("crisis", "crisis", "calm"), ("highvol", "high_vol", "low_vol")]:
        for val, lab in [(1, hi), (0, lo)]:
            s = dd.loc[dd[col] == val, "cv"]
            if len(s) >= 3:
                print("   %-9s n=%3d  mean cond vol (annualized): %.1f%%"
                      % (lab, len(s), np.sqrt(12) * s.mean()))

    # ---- 5. Do fundamentals explain the VARIANCE? ----
    print("\n" + "=" * 70)
    print("5. Do fundamentals (useless for the mean) explain the VARIANCE?")
    print("=" * 70)
    reg = dat.dropna(subset=["cv", "abs_stor_anom", "abs_hdd_anom"]).copy()
    reg["crisis"] = ((reg["Date"] >= CRISIS_START) & (reg["Date"] < CRISIS_END)).astype(float)
    X = sm.add_constant(reg[["abs_stor_anom", "abs_hdd_anom", "crisis"]])
    ols = sm.OLS(np.log(reg["cv"]), X).fit(cov_type="HAC", cov_kwds={"maxlags": 6})
    print("   OLS: log(conditional vol) ~ |storage anom| + |HDD anom| + crisis dummy")
    for c in ["abs_stor_anom", "abs_hdd_anom", "crisis", "const"]:
        print("     %-14s coef=%+.4f  t=%+.2f  p=%.3f" % (c, ols.params[c], ols.tvalues[c], ols.pvalues[c]))
    print("   R2=%.3f  (n=%d)" % (ols.rsquared, int(ols.nobs)))

    # ---- 6. OOS variance forecast: GARCH vs EWMA vs rolling ----
    print("\n" + "=" * 70)
    print("6. Out-of-sample 1-step variance forecast (QLIKE & MSE; lower=better)")
    print("=" * 70)
    rv = r.values
    realized = rv ** 2                                   # realized variance proxy
    ewma = ewma_pred_var(rv)
    roll = rolling_pred_var(rv)
    garch = np.full(len(rv), np.nan)
    for t in range(OOS_MIN_TRAIN, len(rv)):
        try:
            res_t = fit_spec(pd.Series(rv[:t]), vol="GARCH", p=1, o=0, q=1, dist="t")
            fc = res_t.forecast(horizon=1, reindex=False)
            garch[t] = np.asarray(fc.variance.values, float)[-1, 0]
        except Exception:
            garch[t] = np.nan
    idx = slice(OOS_MIN_TRAIN, len(rv))
    R, G, E, Ro = realized[idx], garch[idx], ewma[idx], roll[idx]
    n_oos = np.sum(~np.isnan(G))
    print("   OOS window: %s .. %s  (n=%d)"
          % (dat["Date"].iloc[OOS_MIN_TRAIN].date(), dat["Date"].iloc[-1].date(), int(n_oos)))
    print("   %-14s %10s %10s" % ("Model", "QLIKE", "MSE"))
    for name, P in [("GARCH(1,1)-t", G), ("EWMA(0.94)", E), ("rolling-%d" % ROLL_WIN, Ro)]:
        print("   %-14s %10.4f %10.1f" % (name, qlike(R, P), mse(R, P)))
    # DM on QLIKE differentials vs GARCH (positive => GARCH better)
    def qlike_vec(realized, pred):
        out = np.full(len(pred), np.nan); m = (~np.isnan(pred)) & (pred > 0)
        out[m] = np.log(pred[m]) + realized[m] / pred[m]; return out
    lg, le, lr = qlike_vec(R, G), qlike_vec(R, E), qlike_vec(R, Ro)
    print("   DM t, EWMA vs GARCH  (positive => GARCH better): %.2f" % dm_tstat(le, lg))
    print("   DM t, rolling vs GARCH (positive => GARCH better): %.2f" % dm_tstat(lr, lg))

    print("\nNOTE: monthly GARCH on ~136 obs is data-hungry; magnitudes indicative.")
    print("Conditional-vol series is best.conditional_volatility if you want to plot it.")

In [65]:
# ---- run this section ----
main()

TTF GARCH volatility study.  Sample 2015-02-01..2026-05-01  (136 monthly returns, percent).

1. ARCH-LM test on returns (H0: no ARCH effect)
   LM stat=31.72  p=0.0015   F=3.18  p=0.0006  -> ARCH effect present (GARCH warranted)

2. Volatility-model comparison (mean=Constant)
   Spec                 logL       AIC       BIC     persist half-life
   ARCH(1)-N         -574.55   1155.10   1163.83       0.126    0.3 mo
   GARCH(1,1)-N      -556.37   1120.74   1132.39       1.000       inf
   GARCH(1,1)-t      -555.36   1120.71   1135.28       1.000       inf
   GJR(1,1)-t        -555.29   1122.58   1140.06       1.000       inf
   EGARCH(1,1)-t     -555.40   1122.80   1140.28       0.952   14.0 mo

   Best by BIC: GARCH(1,1)-N

3. Best model (GARCH(1,1)-N) -- parameters & diagnostics
                     Constant Mean - GARCH Model Results                      
Dep. Variable:                      r   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squa

---

## Section 7.2 — Nonlinearity check (gradient boosting)

Source: `ttf_nonlinearity.py`  *(data path normalized to NG_m_final.csv)*

<details><summary>original module docstring</summary>

```
ttf_nonlinearity.py

Curated-subset nonlinearity check for month-ahead (H=1) TTF returns.

Question: on the SAME curated core predictors the linear econometric models use
(lagged momentum, storage-change anomaly, HDD anomaly), does allowing
nonlinearity / interactions add anything the linear core misses?

Two tests, both on the curated 3-predictor set (never the full 55):
  (A) Parametric: add quadratic + pairwise-interaction + crisis-interaction terms
      to the linear core; Newey-West (HAC) JOINT WALD test that they are all zero.
  (B) Nonparametric: a deliberately-regularised SHALLOW gradient boosting on the
      three predictors; expanding walk-forward OOS vs the linear core and a random
      walk (Campbell-Thompson OOS R2 + Diebold-Mariano).

No statsmodels/xgboost needed: HAC + Wald done in numpy; boosting via sklearn.
```

</details>

In [66]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

DATA = "NG_m_final.csv"        # original source used "NG_m_final_full.csv" -- normalized for the combined notebook
CRISIS = ("2021-09-01", "2023-07-01")     # paper's imposed crisis window
NW_LAG = 4                                 # Newey-West Bartlett lag (monthly)
MIN_TRAIN = 72                             # expanding walk-forward start
RNG = 0


# ---------- data + no-look-ahead deseasonalised anomalies ----------
def load():
    df = pd.read_csv(DATA)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    # clean the 2017 JKM constant-fill as in the other scripts (harmless here)
    jkm = df["JKM(USD/mmbtu)"].astype(float).copy()
    jkm[np.isclose(jkm, 7.86)] = np.nan
    df["JKM(USD/mmbtu)"] = np.exp(np.log(jkm).interpolate(limit_direction="both"))
    return df

In [67]:
def deseasonalise(series, dates):
    """raw minus expanding, prior-years-only calendar-month mean (no look-ahead)."""
    s = pd.Series(np.asarray(series, float), index=range(len(series)))
    months = pd.DatetimeIndex(dates).month
    years = pd.DatetimeIndex(dates).year
    out = np.full(len(s), np.nan)
    for i in range(len(s)):
        m, y = months[i], years[i]
        prior = [s[j] for j in range(i) if months[j] == m and years[j] < y]
        prior = [v for v in prior if not np.isnan(v)]
        if prior:
            clim = np.mean(prior)
        else:
            prior_any = s[:i].dropna()
            clim = prior_any.mean() if len(prior_any) else np.nan
        out[i] = s[i] - clim
    return out


def build():
    df = load()
    dates = df["Date"]
    logttf = np.log(df["TTF(USD/mmbtu)"].astype(float).values)
    r = np.concatenate([[np.nan], np.diff(logttf)])          # r_t = dlog TTF_t
    stor_col = "EU+UK Act_storage_change" if "EU+UK Act_storage_change" in df else "EU+UK Calc_storage_change"
    a_mom  = deseasonalise(r, dates)                          # deseasonalised own return
    a_stor = deseasonalise(df[stor_col].astype(float).values, dates)
    a_hdd  = deseasonalise(df["Europe_HDD"].astype(float).values, dates)
    d = pd.DataFrame({"Date": dates, "y": r,
                      "mom": pd.Series(a_mom).shift(1),       # all predictors lagged one month
                      "stor": pd.Series(a_stor).shift(1),
                      "hdd": pd.Series(a_hdd).shift(1)})
    cr = ((d["Date"] >= CRISIS[0]) & (d["Date"] < CRISIS[1])).astype(float)
    d["crisis"] = cr
    d = d.dropna().reset_index(drop=True)
    # standardise the 3 core predictors (train-agnostic here; fine for in-sample Wald)
    for c in ["mom", "stor", "hdd"]:
        d[c] = (d[c] - d[c].mean()) / d[c].std()
    return d, stor_col


# ---------- OLS + Newey-West HAC + joint Wald ----------

In [68]:
def ols_hac(X, y, L=NW_LAG):
    n, k = X.shape
    XtX_inv = np.linalg.inv(X.T @ X)
    beta = XtX_inv @ X.T @ y
    u = y - X @ beta
    S = (X * u[:, None]).T @ (X * u[:, None])                 # lag 0
    for l in range(1, L + 1):
        w = 1.0 - l / (L + 1.0)
        Xl = (X * u[:, None])
        G = Xl[l:].T @ Xl[:-l]
        S += w * (G + G.T)
    cov = XtX_inv @ S @ XtX_inv
    return beta, cov, u


def joint_wald(beta, cov, idx):
    """Wald that the coefficients at positions `idx` are jointly zero."""
    R = np.zeros((len(idx), len(beta)))
    for r_, j in enumerate(idx):
        R[r_, j] = 1.0
    Rb = R @ beta
    W = float(Rb.T @ np.linalg.inv(R @ cov @ R.T) @ Rb)
    q = len(idx)
    p_chi2 = 1 - stats.chi2.cdf(W, q)
    return W, q, p_chi2

In [69]:
def design(d, terms):
    cols = {"const": np.ones(len(d)),
            "mom": d["mom"].values, "stor": d["stor"].values, "hdd": d["hdd"].values,
            "stor2": d["stor"].values**2, "mom2": d["mom"].values**2, "hdd2": d["hdd"].values**2,
            "stor_hdd": d["stor"].values*d["hdd"].values,
            "stor_mom": d["stor"].values*d["mom"].values,
            "hdd_mom": d["hdd"].values*d["mom"].values,
            "stor_cr": d["stor"].values*d["crisis"].values,
            "mom_cr": d["mom"].values*d["crisis"].values,
            "hdd_cr": d["hdd"].values*d["crisis"].values}
    X = np.column_stack([cols[t] for t in terms])
    return X


# ---------- walk-forward OOS ----------
def dm_stat(e1, e2, L=NW_LAG):
    """DM: positive => model-2 better (lower loss). e are squared errors."""
    dloss = e1 - e2
    dbar = dloss.mean()
    n = len(dloss)
    g0 = np.var(dloss)
    var = g0
    for l in range(1, L + 1):
        w = 1 - l / (L + 1.0)
        c = np.cov(dloss[l:], dloss[:-l])[0, 1]
        var += 2 * w * c
    se = np.sqrt(var / n)
    return dbar / se if se > 0 else np.nan

In [70]:
def ct_r2(actual, pred):
    """Campbell-Thompson OOS R2 vs random walk (pred 0)."""
    sse = np.sum((actual - pred) ** 2)
    sst = np.sum(actual ** 2)                                  # RW forecast of return = 0
    return 1 - sse / sst


def walk_forward(d):
    core = ["const", "mom", "stor", "hdd"]
    ext = core + ["stor2", "mom2", "hdd2", "stor_hdd", "stor_mom", "hdd_mom",
                  "stor_cr", "mom_cr", "hdd_cr"]
    rows = []
    for i in range(MIN_TRAIN, len(d)):
        tr, te = d.iloc[:i], d.iloc[i:i+1]
        y = tr["y"].values
        Xc_tr = design(tr, core); Xc_te = design(te, core)
        b = np.linalg.lstsq(Xc_tr, y, rcond=None)[0]
        pred_core = float((Xc_te @ b)[0])
        Xe_tr = design(tr, ext); Xe_te = design(te, ext)
        be = np.linalg.lstsq(Xe_tr, y, rcond=None)[0]
        pred_ext = float((Xe_te @ be)[0])
        Xg_tr = tr[["mom", "stor", "hdd"]].values
        Xg_te = te[["mom", "stor", "hdd"]].values
        gb = GradientBoostingRegressor(max_depth=2, n_estimators=150, learning_rate=0.05,
                                       subsample=0.7, min_samples_leaf=12, random_state=RNG)
        gb.fit(Xg_tr, y)
        pred_gb = float(gb.predict(Xg_te)[0])
        rows.append({"Date": te["Date"].values[0], "actual": float(te["y"].values[0]),
                     "pred_core": pred_core, "pred_ext": pred_ext, "pred_gb": pred_gb,
                     "pred_rw": 0.0, "pred_hist": y.mean()})
    return pd.DataFrame(rows)


# ---------- run ----------

In [71]:
def main():
    d, stor_col = build()
    print("Curated-subset nonlinearity check")
    print("storage column: %s" % stor_col)
    print("usable rows: %d   window %s..%s\n" % (len(d), d["Date"].min().date(), d["Date"].max().date()))
    # --- (A) parametric joint Wald, in-sample on the training window (pre-2025) ---
    tr = d[d["Date"] < "2025-01-01"].reset_index(drop=True)
    core = ["const", "mom", "stor", "hdd"]
    nl_quad_int = ["stor2", "mom2", "hdd2", "stor_hdd", "stor_mom", "hdd_mom"]
    nl_crisis   = ["stor_cr", "mom_cr", "hdd_cr"]
    full_terms = core + nl_quad_int + nl_crisis
    X = design(tr, full_terms); y = tr["y"].values
    beta, cov, u = ols_hac(X, y)
    # positions of the nonlinear blocks in full_terms
    pos = {t: k for k, t in enumerate(full_terms)}
    idx_all = [pos[t] for t in nl_quad_int + nl_crisis]
    idx_quad = [pos[t] for t in nl_quad_int]
    idx_cris = [pos[t] for t in nl_crisis]
    for name, idx in [("quadratics + pairwise interactions (6 df)", idx_quad),
                      ("crisis interactions (3 df)", idx_cris),
                      ("ALL nonlinear terms (9 df)", idx_all)]:
        W, q, p = joint_wald(beta, cov, idx)
        F = W / q
        print("  Joint Wald  %-42s  W=%6.2f  df=%d  F=%5.2f  p=%.3f  %s"
              % (name, W, q, F, p, "REJECT" if p < 0.05 else "no nonlinearity"))
    # in-sample R2 gain
    def r2(terms):
        Xx = design(tr, terms); bb = np.linalg.lstsq(Xx, y, rcond=None)[0]
        e = y - Xx @ bb; return 1 - np.sum(e**2)/np.sum((y-y.mean())**2)
    print("  In-sample R2:  linear core=%.3f   +quad/int=%.3f   +all nonlinear=%.3f"
          % (r2(core), r2(core+nl_quad_int), r2(full_terms)))
    # --- (B) walk-forward OOS: shallow GB vs linear core vs RW ---
    res = walk_forward(d)
    full = res
    post = res[res["Date"] >= "2025-01-01"]
    print("\n  Walk-forward OOS (expanding, refit each month), full window %s..%s (n=%d)"
          % (full["Date"].min().date(), full["Date"].max().date(), len(full)))
    for label, sub in [("FULL window", full), ("post-2025 test", post)]:
        a = sub["actual"].values
        print("   %-16s  n=%2d   OOS R2 vs RW:  core=%+.3f  ext-linear=%+.3f  GB=%+.3f  hist=%+.3f"
              % (label, len(sub), ct_r2(a, sub["pred_core"].values),
                 ct_r2(a, sub["pred_ext"].values),
                 ct_r2(a, sub["pred_gb"].values), ct_r2(a, sub["pred_hist"].values)))
    a = full["actual"].values
    e_core = (a - full["pred_core"].values)**2
    e_ext  = (a - full["pred_ext"].values)**2
    e_gb   = (a - full["pred_gb"].values)**2
    e_rw   = (a - full["pred_rw"].values)**2
    print("   DM  ext-linear vs core (positive => ext better): %+.2f" % dm_stat(e_core, e_ext))
    print("   DM  GB vs core         (positive => GB better):  %+.2f" % dm_stat(e_core, e_gb))
    print("   DM  GB vs RW           (positive => GB better):  %+.2f" % dm_stat(e_rw, e_gb))
    print("   DM  core vs RW         (positive => core better):%+.2f" % dm_stat(e_rw, e_core))

In [72]:
# ---- run this section ----
main()

Curated-subset nonlinearity check
storage column: EU+UK Act_storage_change
usable rows: 134   window 2015-04-01..2026-05-01

  Joint Wald  quadratics + pairwise interactions (6 df)   W= 18.99  df=6  F= 3.17  p=0.004  REJECT
  Joint Wald  crisis interactions (3 df)                  W= 11.09  df=3  F= 3.70  p=0.011  REJECT
  Joint Wald  ALL nonlinear terms (9 df)                  W= 40.69  df=9  F= 4.52  p=0.000  REJECT
  In-sample R2:  linear core=0.251   +quad/int=0.298   +all nonlinear=0.376

  Walk-forward OOS (expanding, refit each month), full window 2021-04-01..2026-05-01 (n=62)
   FULL window       n=62   OOS R2 vs RW:  core=+0.106  ext-linear=-21.949  GB=+0.121  hist=-0.009
   post-2025 test    n=17   OOS R2 vs RW:  core=-0.273  ext-linear=-0.464  GB=-0.268  hist=-0.004
   DM  ext-linear vs core (positive => ext better): -1.06
   DM  GB vs core         (positive => GB better):  +0.20
   DM  GB vs RW           (positive => GB better):  +1.00
   DM  core vs RW         (positive =>

---

## Section 7.3–7.4 — Machine learning: ridge + PCR / PLS

Source: `Gas_monthly_ML.py`

<details><summary>original module docstring</summary>

```
Gas_monthly_ML.py

Consolidated MACHINE-LEARNING / regularisation robustness checks for the
month-ahead (H=1) TTF study -- the umbrella paper's Section 7. It merges the two
scikit-learn scripts into one file and runs them in sequence:

  §7.3  RIDGE / ELASTIC-NET   [was ttf_ridge.py]
        L2 (and, as a secondary lens, elastic-net) shrinkage over the WHOLE
        candidate pool, with no manual selection -- does regularising the full
        pool find forecastable signal the hand-rolled OLS selection missed, or is
        the signal genuinely weak? The selected penalty lambda itself measures
        how much signal there is (lambda -> infinity collapses to the mean/RW).

  §7.4  PCR / PLS             [was ttf_pcr_standalone.py]
        Is there a low-dimensional LATENT-FACTOR structure in the ~55 collinear
        predictors that forecasts even when no individual predictor does?
          - PCR: PCA on standardized predictors, regress y on first k PCs (unsupervised).
          - PLS: components chosen to maximise covariance with y (supervised; fairer).

Both share ONE feature pipeline (defined once below): no-look-ahead deseasonalized
anomalies over the SAME four specifications -- the 2x2 of the headline results:
  1. EU deployable      -- all European predictors, LAGGED
  2. Global deployable  -- all EU + global predictors, LAGGED
  3. EU ceiling         -- EU fundamentals CONTEMPORANEOUS (perfect foresight), momentum lagged
  4. Global ceiling     -- EU + global FUNDAMENTALS contemporaneous; PRICES (JKM/HH/
                           spreads/VIX/FX) and momentum LAGGED (foresight is for quantities,
                           never prices).

Discipline (both models): z-score standardisation fit on the TRAINING window only
(inside a pipeline); the hyperparameter (ridge lambda / #components) chosen by
TimeSeriesSplit CV, re-selected each step of an expanding walk-forward (no
hyperparameter look-ahead); benchmarks are the random walk (0), the historical
mean, and the honest OLS core (momentum + storage_change + HDD, all lagged);
Diebold-Mariano tests vs the random walk and vs the core.

Run:  python Gas_monthly_ML.py   (runs the ridge check, then the PCR/PLS check)

Requirements: scikit-learn, numpy, pandas   (NO statsmodels needed)
```

</details>

In [73]:
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV, ElasticNetCV, LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

In [74]:
# --------------------------------------------------------------------------- #
# Shared configuration
# --------------------------------------------------------------------------- #

DATA_PATH = "NG_m_final.csv"
SAMPLE_START = "2015-01-01"
TRAIN_END = "2025-01-01"
MIN_TRAIN = 60
HORIZON = 1
CV_SPLITS = 5

# Ridge / elastic-net grids
ALPHAS = np.logspace(-2, 5, 40)          # ridge lambda grid
L1_RATIOS = [0.1, 0.5, 0.9]              # elastic-net mix grid
# PCR / PLS grid
K_GRID = [1, 2, 3, 5, 8, 12]

SPECS = [("eu_deploy", "EU deployable (all lagged)"),
         ("global_deploy", "Global deployable (all lagged)"),
         ("eu_ceiling", "EU ceiling (fundamentals foreseen)"),
         ("global_ceiling", "Global ceiling (quantities foreseen, prices lagged)")]

In [75]:
# --------------------------------------------------------------------------- #
# Shared: data + no-look-ahead anomaly builders
# --------------------------------------------------------------------------- #

def load_data(path=DATA_PATH):
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    df["month"] = df["Date"].dt.month
    df["TTF"] = df["TTF(USD/mmbtu)"]
    # JKM: replace the 4-month (Jan-Apr 2017) constant fill at 7.86 with interpolation
    if "JKM(USD/mmbtu)" in df.columns:
        jkm = df["JKM(USD/mmbtu)"].astype(float).copy()
        jkm[np.isclose(jkm, 7.86)] = np.nan
        df["JKM(USD/mmbtu)"] = np.exp(np.log(jkm).interpolate(limit_direction="both"))
        df["ttf_jkm_logspread"] = np.log(df["TTF(USD/mmbtu)"]) - np.log(df["JKM(USD/mmbtu)"])
        df["ttf_hh_logspread"] = np.log(df["TTF(USD/mmbtu)"]) - np.log(df["HH(USD/mmbtu)"])
    return df.loc[df["Date"] >= SAMPLE_START].reset_index(drop=True)


def expanding_seasonal_mean(series, month):
    out = pd.Series(index=series.index, dtype=float)
    tmp = pd.DataFrame({"val": series, "month": month})
    for _, grp in tmp.groupby("month"):
        out.loc[grp.index] = grp["val"].expanding().mean().shift(1)
    return out

In [76]:
def level_anom(df, col):
    return df[col] - expanding_seasonal_mean(df[col], df["month"])


def change_anom(df, col):
    ch = df[col].diff()
    return ch - expanding_seasonal_mean(ch, df["month"])


def dlog_anom(df, col):
    dl = np.log(df[col]).diff()
    return dl - expanding_seasonal_mean(dl, df["month"])


TRANSFORMS = {"level_anom": level_anom, "change_anom": change_anom, "dlog_anom": dlog_anom}

# (name, source col, transform, block)  -- block in {"Own","EU","Global"}
CANDIDATES = [
    ("momentum",                 "TTF",                        "dlog_anom",  "Own"),
    # ---- European block ----
    ("storage_change_anom",      "EU+UK_av_storage(bcm)",      "change_anom", "EU"),
    ("storage_level_anom",       "EU+UK_av_storage(bcm)",      "level_anom",  "EU"),
    ("production_anom",          "EU+UK Production(bcm)",       "level_anom",  "EU"),
    ("net_piped_anom",           "EU+UK Net_piped(bcm)",       "level_anom",  "EU"),
    ("lng_imports_anom",         "EU+UK LNG imports",          "level_anom",  "EU"),
    ("net_supply_anom",          "EU+UK Net_supply",           "level_anom",  "EU"),
    ("total_demand_anom",        "EU+UK Total(bcm)",           "level_anom",  "EU"),
    ("nonpower_demand_anom",     "EU+UK Non_power(bcm)",        "level_anom",  "EU"),
    ("power_gas_demand_anom",    "EU+UK Electricity(bcm)",     "level_anom",  "EU"),
    ("residual_load_anom",       "EU+UK Residual load",        "level_anom",  "EU"),
    ("coal_gen_anom",            "EU+UK Coal",                 "level_anom",  "EU"),
    ("nuclear_gen_anom",         "EU+UK Nuclear",              "level_anom",  "EU"),
    ("hydro_gen_anom",           "EU+UK Hydro_gen",            "level_anom",  "EU"),
    ("gas_burn_anom",            "EU+UK Fossil gas",           "level_anom",  "EU"),
    ("production_chg_anom",      "EU+UK Production(bcm)",       "change_anom", "EU"),
    ("net_piped_chg_anom",       "EU+UK Net_piped(bcm)",       "change_anom", "EU"),
    ("lng_imports_chg_anom",     "EU+UK LNG imports",          "change_anom", "EU"),
    ("net_supply_chg_anom",      "EU+UK Net_supply",           "change_anom", "EU"),
    ("total_demand_chg_anom",    "EU+UK Total(bcm)",           "change_anom", "EU"),
    ("nonpower_demand_chg_anom", "EU+UK Non_power(bcm)",        "change_anom", "EU"),
    ("power_gas_demand_chg_anom","EU+UK Electricity(bcm)",     "change_anom", "EU"),
    ("residual_load_chg_anom",   "EU+UK Residual load",        "change_anom", "EU"),
    ("coal_gen_chg_anom",        "EU+UK Coal",                 "change_anom", "EU"),
    ("nuclear_gen_chg_anom",     "EU+UK Nuclear",              "change_anom", "EU"),
    ("hydro_gen_chg_anom",       "EU+UK Hydro_gen",            "change_anom", "EU"),
    ("gas_burn_chg_anom",        "EU+UK Fossil gas",           "change_anom", "EU"),
    ("norway_prod_anom",         "Norway_gas_prod",            "level_anom",  "EU"),
    ("norway_supplyred_anom",    "Norway_supply_red",          "level_anom",  "EU"),
    ("norway_planned_anom",      "Norway_planned_outage",      "level_anom",  "EU"),
    ("norway_unplanned_anom",    "Norway_unplanned_outage",    "level_anom",  "EU"),
    ("hdd_anom",                 "Europe_HDD",                 "level_anom",  "EU"),
    ("cdd_anom",                 "Europe_CDD",                 "level_anom",  "EU"),
    ("wind_anom",                "EU_wind_speed",              "level_anom",  "EU"),
    ("solar_anom",               "EU_solar",                   "level_anom",  "EU"),
    ("precip_anom",              "Nordic_precip",              "level_anom",  "EU"),
    # ---- Global block ----
    ("jkm_mom",                  "JKM(USD/mmbtu)",             "dlog_anom",   "Global"),
    ("hh_mom",                   "HH(USD/mmbtu)",              "dlog_anom",   "Global"),
    ("ttf_jkm_spread_anom",      "ttf_jkm_logspread",          "level_anom",  "Global"),
    ("ttf_hh_spread_anom",       "ttf_hh_logspread",           "level_anom",  "Global"),
    ("ttf_jkm_spread_chg_anom",  "ttf_jkm_logspread",          "change_anom", "Global"),
    ("us_gwdd_anom",             "US_GWDD",                    "level_anom",  "Global"),
    ("neasia_gwdd_anom",         "NE_Asia_GWDD",               "level_anom",  "Global"),
    ("atlantic_ace_anom",        "Atlantic_ACE",               "level_anom",  "Global"),
    ("gulf_storm_anom",          "Gulf_storm_days",            "level_anom",  "Global"),
    ("global_lng_offline_anom",  "Global LNG capacity offline","level_anom",  "Global"),
    ("global_lng_offline_chg_anom","Global LNG capacity offline","change_anom","Global"),
    ("global_lng_capacity_chg_anom","Global LNG nameplate capacity","change_anom","Global"),
    ("asia_imports_chg_anom",    "CH+JP+KR LNG imports",       "change_anom", "Global"),
    ("india_imports_chg_anom",   "IN LNG imports",             "change_anom", "Global"),
    ("qaus_exports_chg_anom",    "QA+AU+US LNG exports",       "change_anom", "Global"),
    ("seasia_exports_chg_anom",  "ID+MY+BN LNG exports",       "change_anom", "Global"),
    ("nigeria_exports_chg_anom", "NG LNG exports",             "change_anom", "Global"),
    ("vix_anom",                 "VIX",                        "level_anom",  "Global"),
    ("fx_ret_anom",              "USD-EUR_FX",                 "dlog_anom",   "Global"),
]

# Prices / financial / own-return: NEVER given foresight (always lagged, even in ceilings)

In [77]:
ALWAYS_LAG = {"momentum", "jkm_mom", "hh_mom", "ttf_jkm_spread_anom",
              "ttf_hh_spread_anom", "ttf_jkm_spread_chg_anom", "vix_anom", "fx_ret_anom"}

CORE = ["momentum__lag", "storage_change_anom__lag", "hdd_anom__lag"]


def build_features(df, horizon=HORIZON):
    """No-look-ahead anomalies; every predictor gets a __lag column and (unless a
    price/financial/own-return term) a contemporaneous __now column."""
    pieces = {"Date": df["Date"], "y": np.log(df["TTF"]).diff(horizon)}
    for name, col, tr, block in CANDIDATES:
        if col not in df.columns:
            continue
        anom = TRANSFORMS[tr](df, col)
        pieces[name + "__lag"] = anom.shift(horizon)          # predetermined
        if name not in ALWAYS_LAG:
            pieces[name + "__now"] = anom                      # contemporaneous (foresight)
    return pd.DataFrame(pieces)          # built at once -> no fragmentation warning


def spec_cols(which):
    eu = [n for n, c, t, b in CANDIDATES if b in ("Own", "EU")]
    alln = [n for n, c, t, b in CANDIDATES]
    if which == "eu_deploy":
        return [n + "__lag" for n in eu]
    if which == "global_deploy":
        return [n + "__lag" for n in alln]
    if which == "eu_ceiling":
        return [(n + "__lag" if n in ALWAYS_LAG else n + "__now") for n in eu]
    if which == "global_ceiling":
        return [(n + "__lag" if n in ALWAYS_LAG else n + "__now") for n in alln]
    raise ValueError(which)

In [78]:
def dm_tstat(loss_worse, loss_better):
    d = np.asarray(loss_worse, float) - np.asarray(loss_better, float)
    d = d[~np.isnan(d)]
    n = len(d); s = d.std(ddof=1)
    return d.mean() / (s / np.sqrt(n)) if s > 0 else np.nan


def summarize(res, names):
    rw_err = res["actual"] - res["pred_rw"]
    out = []
    for nm in names:
        pred = res["pred_" + nm]; err = res["actual"] - pred
        out.append({
            "Model": nm,
            "RMSE": np.sqrt((err ** 2).mean()),
            "Hit%": np.nan if nm == "rw" else 100 * (np.sign(res["actual"]) == np.sign(pred)).mean(),
            "OOS_R2_vs_RW": np.nan if nm == "rw" else 1 - (err ** 2).sum() / (rw_err ** 2).sum(),
            "DM_vs_RW": np.nan if nm == "rw" else dm_tstat((res["actual"] - res["pred_rw"]) ** 2, err ** 2),
        })
    return pd.DataFrame(out)


# =========================================================================== #
# §7.3  RIDGE / ELASTIC-NET   (was ttf_ridge.py)
# =========================================================================== #

In [79]:
def ridge_walk_forward(feat, cols, min_train=MIN_TRAIN):
    keep = ["Date", "y"] + sorted(set(cols) | set(CORE))
    keep = [c for c in keep if c in feat.columns]
    core = [c for c in CORE if c in feat.columns]
    data = feat[keep].dropna().reset_index(drop=True)
    rows = []
    for i in range(min_train, len(data)):
        tr, te = data.iloc[:i], data.iloc[i:i + 1]
        ytr = tr["y"].values
        Xtr, Xte = tr[cols].values, te[cols].values
        nsp = int(min(CV_SPLITS, max(2, i // 12)))
        tscv = TimeSeriesSplit(n_splits=nsp)
        rid = make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS, cv=tscv)).fit(Xtr, ytr)
        en = make_pipeline(StandardScaler(),
                           ElasticNetCV(l1_ratio=L1_RATIOS, alphas=ALPHAS, cv=tscv, max_iter=5000)
                           ).fit(Xtr, ytr)
        rec = {"Date": te["Date"].values[0], "actual": te["y"].values[0],
               "pred_rw": 0.0, "pred_hist": ytr.mean(),
               "pred_ridge": rid.predict(Xte)[0], "pred_enet": en.predict(Xte)[0],
               "alpha_ridge": rid.named_steps["ridgecv"].alpha_}
        if core:
            lr = LinearRegression().fit(tr[core].values, ytr)
            rec["pred_core"] = lr.predict(te[core].values)[0]
        else:
            rec["pred_core"] = np.nan
        rows.append(rec)
    return pd.DataFrame(rows), data

In [80]:
def ridge_top_coefs(feat, cols, k=6):
    """Full-training-window standardized ridge coefs + elastic-net sparsity."""
    d = feat[["y"] + cols].loc[feat["Date"] < TRAIN_END].dropna()
    X, y = d[cols].values, d["y"].values
    tscv = TimeSeriesSplit(n_splits=CV_SPLITS)
    rid = make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS, cv=tscv)).fit(X, y)
    en = make_pipeline(StandardScaler(),
                       ElasticNetCV(l1_ratio=L1_RATIOS, alphas=ALPHAS, cv=tscv, max_iter=5000)).fit(X, y)
    rc = pd.Series(rid.named_steps["ridgecv"].coef_, index=cols).sort_values(key=np.abs, ascending=False)
    ec = pd.Series(en.named_steps["elasticnetcv"].coef_, index=cols)
    nz = ec[ec.abs() > 1e-8].sort_values(key=np.abs, ascending=False)
    return rid.named_steps["ridgecv"].alpha_, rc.head(k), len(nz), nz.head(k)


def run_ridge():
    print("\n\n" + "#" * 78)
    print("# §7.3  RIDGE / ELASTIC-NET robustness check")
    print("#" * 78)
    df = load_data()
    feat = build_features(df)
    print("Ridge / elastic-net robustness check.  Sample %s..%s (n=%d rows before dropna)."
          % (feat["Date"].min().date(), feat["Date"].max().date(), len(feat)))
    print("lambda by TimeSeriesSplit CV; z-score on train only; benchmarks RW / hist-mean / OLS core.\n")

    summary_rows = []
    for key, label in SPECS:
        cols = [c for c in spec_cols(key) if c in feat.columns]
        print("=" * 78)
        print("%s   [%d predictors]" % (label, len(cols)))
        print("=" * 78)
        res, data = ridge_walk_forward(feat, cols)
        print("OOS window %s..%s  (n=%d)"
              % (pd.Timestamp(res["Date"].min()).date(), pd.Timestamp(res["Date"].max()).date(), len(res)))
        tab = summarize(res, ["ridge", "enet", "core", "hist", "rw"])
        print(tab.to_string(index=False, float_format=lambda x: "%.3f" % x))
        print("DM ridge vs core (positive => ridge better): %.2f"
              % dm_tstat((res["actual"] - res["pred_core"]) ** 2, (res["actual"] - res["pred_ridge"]) ** 2))
        amed = np.nanmedian(res["alpha_ridge"])
        print("Selected ridge lambda: median=%.3g  (min=%.3g, max=%.3g)  -- large => little signal"
              % (amed, np.nanmin(res["alpha_ridge"]), np.nanmax(res["alpha_ridge"])))
        alpha_ft, rc, n_nz, ec = ridge_top_coefs(feat, cols)
        print("Full-train ridge, top standardized coefficients:")
        for nm, v in rc.items():
            print("   %-30s %+.4f" % (nm, v))
        print("Elastic-net keeps %d/%d predictors; top nonzero:" % (n_nz, len(cols)))
        for nm, v in ec.items():
            print("   %-30s %+.4f" % (nm, v))
        r = tab.set_index("Model")
        summary_rows.append({"spec": label,
                             "ridge_OOS_R2": r.loc["ridge", "OOS_R2_vs_RW"],
                             "ridge_DM_RW": r.loc["ridge", "DM_vs_RW"],
                             "core_OOS_R2": r.loc["core", "OOS_R2_vs_RW"],
                             "lambda_med": amed})
        print()

    print("=" * 78)
    print("CROSS-SPEC SUMMARY (does regularizing the full pool beat RW / the hand-picked core?)")
    print("=" * 78)
    print(pd.DataFrame(summary_rows).to_string(index=False, float_format=lambda x: "%.3f" % x))
    print("\nRead: ridge OOS R2 vs RW near/below 0 and a large lambda => the full pool holds no")
    print("more month-ahead signal than the parsimonious core; confirms weak signal, not crude selection.")


# =========================================================================== #
# §7.4  PCR / PLS   (was ttf_pcr_standalone.py)
# =========================================================================== #

In [81]:
def _cv(i):
    return TimeSeriesSplit(n_splits=int(min(CV_SPLITS, max(2, i // 12))))


def _fit_pcr(X, y, tscv, kmax):
    grid = [k for k in K_GRID if k <= kmax]
    pipe = Pipeline([("sc", StandardScaler()), ("pca", PCA()), ("lr", LinearRegression())])
    gs = GridSearchCV(pipe, {"pca__n_components": grid}, cv=tscv,
                      scoring="neg_mean_squared_error").fit(X, y)
    return gs.best_estimator_, gs.best_params_["pca__n_components"]


def _fit_pls(X, y, tscv, kmax):
    grid = [k for k in K_GRID if k <= kmax]
    gs = GridSearchCV(PLSRegression(scale=True), {"n_components": grid}, cv=tscv,
                      scoring="neg_mean_squared_error").fit(X, y)
    return gs.best_estimator_, gs.best_params_["n_components"]


def pcr_walk_forward(feat, cols, min_train=MIN_TRAIN):
    keep = ["Date", "y"] + sorted(set(cols) | set(CORE))
    keep = [c for c in keep if c in feat.columns]
    core = [c for c in CORE if c in feat.columns]
    data = feat[keep].dropna().reset_index(drop=True)
    rows = []
    for i in range(min_train, len(data)):
        tr, te = data.iloc[:i], data.iloc[i:i + 1]
        ytr = tr["y"].values
        Xtr, Xte = tr[cols].values, te[cols].values
        tscv = _cv(i)
        kmax = min(len(cols), i - 2)
        pcr, k_pcr = _fit_pcr(Xtr, ytr, tscv, kmax)
        pls, k_pls = _fit_pls(Xtr, ytr, tscv, kmax)
        rec = {"Date": te["Date"].values[0], "actual": te["y"].values[0],
               "pred_rw": 0.0, "pred_hist": ytr.mean(),
               "pred_pcr": float(np.ravel(pcr.predict(Xte))[0]),
               "pred_pls": float(np.ravel(pls.predict(Xte))[0]),
               "k_pcr": k_pcr, "k_pls": k_pls}
        rec["pred_core"] = (LinearRegression().fit(tr[core].values, ytr)
                            .predict(te[core].values)[0]) if core else np.nan
        rows.append(rec)
    return pd.DataFrame(rows)

In [82]:
def pc_variance(feat, cols, k=5):
    d = feat[["y"] + cols].loc[feat["Date"] < TRAIN_END].dropna()
    Xs = StandardScaler().fit_transform(d[cols].values)
    pca = PCA().fit(Xs)
    return np.cumsum(pca.explained_variance_ratio_)[:k]


def run_pcr():
    print("\n\n" + "#" * 78)
    print("# §7.4  PCR / PLS dimension-reduction check")
    print("#" * 78)
    df = load_data()
    feat = build_features(df)
    print("PCR / PLS dimension-reduction check.  Sample %s..%s.  k by TimeSeriesSplit CV.\n"
          % (feat["Date"].min().date(), feat["Date"].max().date()))
    summary = []
    for key, label in SPECS:
        cols = [c for c in spec_cols(key) if c in feat.columns]
        print("=" * 78)
        print("%s   [%d predictors]" % (label, len(cols)))
        print("=" * 78)
        res = pcr_walk_forward(feat, cols)
        print("OOS window %s..%s  (n=%d)"
              % (pd.Timestamp(res["Date"].min()).date(), pd.Timestamp(res["Date"].max()).date(), len(res)))
        tab = summarize(res, ["pcr", "pls", "core", "hist", "rw"])
        print(tab.to_string(index=False, float_format=lambda x: "%.3f" % x))
        print("DM PCR vs core (positive => PCR better): %.2f"
              % dm_tstat((res["actual"] - res["pred_core"]) ** 2, (res["actual"] - res["pred_pcr"]) ** 2))
        print("DM PLS vs core (positive => PLS better): %.2f"
              % dm_tstat((res["actual"] - res["pred_core"]) ** 2, (res["actual"] - res["pred_pls"]) ** 2))
        print("Selected components: PCR median k=%.0f (range %d-%d);  PLS median k=%.0f (range %d-%d)"
              % (np.median(res["k_pcr"]), res["k_pcr"].min(), res["k_pcr"].max(),
                 np.median(res["k_pls"]), res["k_pls"].min(), res["k_pls"].max()))
        cv = pc_variance(feat, cols)
        print("Cumulative PC variance explained (train): " + ", ".join(
              "PC1-%d=%.0f%%" % (j + 1, 100 * v) for j, v in enumerate(cv)))
        r = tab.set_index("Model")
        summary.append({"spec": label, "PCR_OOS_R2": r.loc["pcr", "OOS_R2_vs_RW"],
                        "PLS_OOS_R2": r.loc["pls", "OOS_R2_vs_RW"],
                        "core_OOS_R2": r.loc["core", "OOS_R2_vs_RW"],
                        "PLS_DM_RW": r.loc["pls", "DM_vs_RW"]})
        print()
    print("=" * 78)
    print("CROSS-SPEC SUMMARY (do latent factors beat RW / the hand-picked core?)")
    print("=" * 78)
    print(pd.DataFrame(summary).to_string(index=False, float_format=lambda x: "%.3f" % x))
    print("\nRead: PCR/PLS OOS R2 near/below the core and few selected components => no")
    print("exploitable low-dimensional factor structure beyond what storage+momentum already give.")


# =========================================================================== #
# Master entry point
# =========================================================================== #

In [83]:
def main():
    run_ridge()   # §7.3
    run_pcr()     # §7.4

In [84]:
# ---- run this section ----
main()



##############################################################################
# §7.3  RIDGE / ELASTIC-NET robustness check
##############################################################################
Ridge / elastic-net robustness check.  Sample 2015-01-01..2026-05-01 (n=137 rows before dropna).
lambda by TimeSeriesSplit CV; z-score on train only; benchmarks RW / hist-mean / OLS core.

EU deployable (all lagged)   [36 predictors]
OOS window 2021-03-01..2026-05-01  (n=63)
Model  RMSE   Hit%  OOS_R2_vs_RW  DM_vs_RW
ridge 0.205 44.444        -0.049    -0.485
 enet 0.207 52.381        -0.067    -0.621
 core 0.195 60.317         0.057     0.573
 hist 0.201 49.206        -0.007    -0.246
   rw 0.201    NaN           NaN       NaN
DM ridge vs core (positive => ridge better): -0.95
Selected ridge lambda: median=203  (min=4.92, max=1e+05)  -- large => little signal
Full-train ridge, top standardized coefficients:
   lng_imports_anom__lag          -0.0351
   cdd_anom__lag                  +